# 🧠 Lab 02 — Basics of Neural Networks

**DL2026 · Practical Session 2 · Slide set 02**

Last week you learned the Python and NumPy you need. This week you build **the neuron itself** — from the
1943 logic gate to a hand-written multilayer network that reads handwritten digits.

> 🔑 **The through-line of this whole lab is one neuron.**
> The **perceptron** is that neuron with a bad learning rule, **Adaline** is the same neuron with a good one,
> and the **multilayer network** at the end is that same neuron repeated. After section 1, nothing genuinely
> new is introduced except *the wiring*.

> 💡 **This is a hands-on notebook.** Reading it is not enough. Run every code cell (`Shift + Enter`),
> change the numbers, break things, and do the ✍️ exercises before opening the solutions.

**Rules of the game this week:** everything is **NumPy and pen-and-paper**. No PyTorch, and no
scikit-learn *models* — we only borrow scikit-learn to *load data*. Every gradient in this notebook is one
you could have derived yourself. PyTorch arrives in Lab 03, and it will feel easy precisely because you did
this first.

## 🎯 Learning Outcomes

By the end of this lab you should be able to:

| # | You will be able to… | Section |
|:--|:---------------------|:--------|
| 1 | Explain the artificial neuron as net input + activation, and why the **bias is just the threshold, moved** | [§1](#s1) |
| 2 | Load, inspect and plot the Iris data, and state what **nominal** class labels imply | [§2](#s2) |
| 3 | Apply the **perceptron learning rule** by hand and implement it in NumPy | [§3](#s3) |
| 4 | Say exactly which **one edge moved** turns a perceptron into **Adaline**, and why that gives it a loss | [§4](#s4) |
| 5 | Derive $\partial L/\partial w_j$ for the MSE loss and **verify it numerically** | [§4](#s4) |
| 6 | Standardise features and explain why a single learning rate then works | [§5](#s5) |
| 7 | Compare **full-batch, stochastic and mini-batch GD** per step and per epoch, and use an **adaptive $\eta$** | [§6](#s6) |
| 8 | Prove that a stack of linear layers **collapses** into one, so depth needs non-linearity | [§7](#s7) |
| 9 | Implement forward- and back-propagation from scratch for a **one-hidden-layer network** | [§8](#s8), [§9](#s9) |
| 10 | Compare **activation functions** and their derivatives, and say why the unit step cannot be trained | [§10](#s10) |

<a id="toc"></a>
## 📑 Table of Contents

| Section | Topic | What you practise |
|:--|:--|:--|
| [0](#s0) | **Setup & helpers** | environment check, plotting helpers |
| [1](#s1) | **The artificial neuron** | MCP logic gates, unit step, threshold → bias, decision boundary |
| [2](#s2) | **The Iris dataset** | load, inspect, plot features vs labels, the $X$ / $y$ notation |
| [3](#s3) | **The perceptron learning rule** | updates by hand, a `Perceptron` class, convergence and its limits |
| [4](#s4) | **Adaline & gradient descent** | MSE loss, the derivative, gradient checking, learning rates |
| [5](#s5) | **Standardisation** | why feature scales fight a single $\eta$ |
| [6](#s6) | **Batch vs stochastic GD** | cost per step, `AdalineSGD`, mini-batches, adaptive $\eta$ |
| [7](#s7) | **Why depth needs non-linearity** | the linear collapse, XOR, layer shapes, one-hot |
| [8](#s8) | **ANN from scratch — Iris** | forward, backward, gradient check, training loop |
| [9](#s9) | **ANN from scratch — MNIST** | 784 → 50 → 10, 39,700 weights, mini-batch training |
| [10](#s10) | **Activation functions** | step / linear / sigmoid / tanh / ReLU, derivatives, vanishing gradients |
| [11](#s11) | **Self-check quiz** | 10 questions |
| [12](#s12) | **Summary & cheat sheet** | what to carry into Lab 03 |

**Running this in Google Colab:** `File ▸ Upload notebook`, or open it straight from GitHub with
`File ▸ Open notebook ▸ GitHub`. Everything below runs on the free **CPU** runtime — no GPU needed.
End-to-end runtime is roughly **2–4 minutes**.

---
<a id="s0"></a>
# 0 · Setup & Helpers

[⬆ back to TOC](#toc)

Run the next three cells first. They check your environment and define the few helpers this notebook
reuses. Everything we need is pre-installed in Colab.

In [ ]:
import sys, platform

print("Python :", sys.version.split()[0], "on", platform.system())
print("Colab  :", "yes" if "google.colab" in sys.modules else "no (local Jupyter)")
print()

# Anything reported as MISSING can be installed with:
#   %pip install numpy pandas matplotlib scikit-learn ipywidgets
for name in ["numpy", "pandas", "matplotlib", "sklearn", "ipywidgets"]:
    try:
        module = __import__(name)
        print(f"{name:<12}: {getattr(module, '__version__', 'unknown')}")
    except ImportError:
        print(f"{name:<12}: MISSING")

### 🧪 Demo — The helpers we reuse

* `show(label, value)` — print a value together with its type (from Lab 01).
* `render_mermaid(diagram)` — draw the diagrams of this notebook as images.
* `plot_decision_regions(...)` — colour the plane by what a model predicts. This is *the* plot of the
  week: every model in sections 3–8 is judged by the shape of the boundary it draws.
* `RNG` — one seeded random generator, so your numbers match the text.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(1)          # one seeded generator -> reproducible numbers

plt.rcParams.update({
    "figure.figsize": (6.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})


def show(label, value):
    """Print a labelled value together with its Python type."""
    print(f"{label:<22} = {value!r:<30} (type: {type(value).__name__})")


def render_mermaid(diagram):
    """Render a Mermaid diagram as an image; fall back to printing the source when offline."""
    import base64
    try:
        from IPython.display import Image, display
        payload = base64.urlsafe_b64encode(diagram.strip().encode("utf8")).decode("ascii")
        display(Image(url="https://mermaid.ink/img/" + payload))
    except Exception as error:
        print(f"[diagram not rendered: {error}]\n")
        print(diagram)


def plot_decision_regions(X, y, predict, ax=None, resolution=0.02,
                          xlabel="feature 1", ylabel="feature 2", title=None,
                          class_names=None):
    """
    Colour a 2-D plane by the class a model predicts, then scatter the data on top.

    Args:
        X: Feature matrix, shape (n_samples, 2).
        y: Integer class labels, shape (n_samples,).
        predict: Callable mapping an (n, 2) array to (n,) class labels.
        ax: Optional matplotlib axes to draw on.
        resolution: Grid step of the background mesh.
        xlabel, ylabel, title: Labels for the figure.
        class_names: Optional list of names used in the legend.

    Returns:
        The matplotlib axes that was drawn on.
    """
    if ax is None:
        _, ax = plt.subplots()

    markers = ("o", "s", "^", "v", "D")
    colors = ("#3b6fb6", "#d1495b", "#66a182", "#c98b2e", "#7d5ba6")

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution),
                           np.arange(x2_min, x2_max, resolution))
    grid = np.array([xx1.ravel(), xx2.ravel()]).T          # (n_grid, 2) - same columns as X
    zz = np.asarray(predict(grid)).reshape(xx1.shape)

    classes = np.unique(y)
    n_levels = max(len(classes) - 1, 1)
    ax.contourf(xx1, xx2, zz, alpha=0.25, levels=n_levels, colors=colors[:len(classes)])
    ax.contour(xx1, xx2, zz, colors="k", linewidths=0.8, alpha=0.6, levels=n_levels)

    for index, cls in enumerate(classes):
        label = class_names[index] if class_names is not None else f"class {cls}"
        ax.scatter(X[y == cls, 0], X[y == cls, 1],
                   c=colors[index % len(colors)], marker=markers[index % len(markers)],
                   edgecolor="black", linewidth=0.5, s=45, label=label)

    ax.set_xlim(xx1.min(), xx1.max())
    ax.set_ylim(xx2.min(), xx2.max())
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", framealpha=0.9)
    return ax


show("RNG", RNG)
print("helpers ready: show, render_mermaid, plot_decision_regions")

---
<a id="s1"></a>
# 1 · The Artificial Neuron

[⬆ back to TOC](#toc)

## 1.1 Where the model came from

The biological neuron gives us **vocabulary**, not a model. Two details that textbooks usually skip are
exactly the two that survive into the mathematics:

| Biology | What it becomes |
|:--|:--|
| Dendrites collect signals from other cells | the **inputs** $x_1 \dots x_m$, each with a **weight** |
| The **axon hillock** fires only past a threshold | the **bias** $b$ |
| The spike is **all-or-none** (fires, or does not) | the **unit step** activation |

Two dates matter:

* **1943 — McCulloch & Pitts**: the nerve cell as a *logic gate*. The weights are **given** by hand.
* **1957 — Rosenblatt**: the *perceptron learning rule* laid on top of that neuron — an algorithm that
  **finds** the weights from data. That single step is where machine learning as we practise it begins.

In [ ]:
neuron_diagram = """
flowchart LR
    x1(("x1")) -->|w1| S["sum: z = wᵀx + b"]
    x2(("x2")) -->|w2| S
    xm(("xm")) -->|wm| S
    b(("bias b")) --> S
    S --> A["activation<br/>sigma(z)"]
    A --> Y(("y-hat"))
"""
render_mermaid(neuron_diagram)

## 1.2 The 1943 neuron: weights set by hand

Before any learning, notice that a *single* neuron with hand-chosen weights already computes logic gates.
Take $\sigma(z) = 1$ if $z \ge 0$ else $0$, and

$$z = w_1x_1 + w_2x_2 + b.$$

* **AND** — fire only when both inputs are 1: $\mathbf{w} = (1, 1),\; b = -1.5$
* **OR** — fire when at least one is 1: $\mathbf{w} = (1, 1),\; b = -0.5$

The bias sets *how much evidence is enough*. Same weights, different bias, different gate.

In [ ]:
import numpy as np

def unit_step(z):
    """The all-or-none activation: 1 if z >= 0, else 0."""
    return np.where(z >= 0.0, 1, 0)


def neuron(X, w, b):
    """One McCulloch-Pitts neuron: net input, then the unit step."""
    z = X @ w + b               # (n, m) @ (m,) -> (n,), then broadcast the scalar bias
    return unit_step(z)


INPUTS = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])     # every 2-bit pattern

GATES = {
    "AND":  (np.array([1.0, 1.0]), -1.5),
    "OR":   (np.array([1.0, 1.0]), -0.5),
    "NAND": (np.array([-1.0, -1.0]), 1.5),
    "NOR":  (np.array([-1.0, -1.0]), 0.5),
}

header = "x1  x2 | " + "  ".join(f"{name:>4}" for name in GATES)
print(header)
print("-" * len(header))
for x1, x2 in INPUTS:
    outputs = [neuron(np.array([[x1, x2]]), w, b)[0] for w, b in GATES.values()]
    print(f" {x1}   {x2} | " + "  ".join(f"{o:>4}" for o in outputs))

print("\nAND and NAND use the same |weights| and differ only in sign and bias.")

## 1.3 Threshold → bias: the same neuron, rewritten

The original formulation compares the net input against a threshold $\theta$:

$$\sigma(z) = \begin{cases} 1 & \text{if } z \ge \theta \\ 0 & \text{otherwise}\end{cases}
\qquad\text{with}\qquad z = w_1x_1 + \dots + w_mx_m .$$

Move $\theta$ to the left-hand side and rename $-\theta$ as $b$:

$$z = \mathbf{w}^\top\mathbf{x} + b, \qquad \sigma(z) = 1 \text{ if } z \ge 0 .$$

**The bias is not a new idea — it is the threshold, moved.** This is the `bias=True` argument you will
meet in every framework from Lab 03 onwards. The cell below shows the two forms are the *same function*.

In [ ]:
import numpy as np

w = np.array([2.0, -1.0])
theta = 0.5                       # threshold form
b = -theta                        # bias form:  b = -theta

X_random = RNG.uniform(-3, 3, size=(1000, 2))

with_threshold = (X_random @ w >= theta).astype(int)      # compare against theta
with_bias      = (X_random @ w + b >= 0).astype(int)      # fold theta into the net input

print("identical on all 1000 random points:", np.array_equal(with_threshold, with_bias))
print("disagreements                     :", int((with_threshold != with_bias).sum()))

## 1.4 The decision boundary is a line: weights rotate it, the bias slides it

$\sigma(\mathbf{w}^\top\mathbf{x} + b)$ splits the plane along the line $z = 0$. In 2-D:

$$w_1x_1 + w_2x_2 + b = 0 \quad\Longleftrightarrow\quad x_2 = -\frac{w_1}{w_2}\,x_1 - \frac{b}{w_2}$$

so $\mathbf{w}$ controls the **slope** (it is the normal vector of the line) and $b$ controls the
**offset**. That is the entire model: *one neuron can only ever draw one straight line.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def draw_boundary(ax, w, b, title):
    """Shade where the neuron fires, then draw the z = 0 line and the weight vector."""
    grid_x, grid_y = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
    points = np.c_[grid_x.ravel(), grid_y.ravel()]
    fires = unit_step(points @ np.asarray(w, dtype=float) + b).reshape(grid_x.shape)

    ax.contourf(grid_x, grid_y, fires, levels=1, colors=["#e8eef7", "#f7e3e6"])
    ax.contour(grid_x, grid_y, fires, levels=[0.5], colors="k", linewidths=1.5)
    ax.arrow(0, 0, w[0], w[1], head_width=0.15, color="#d1495b", length_includes_head=True)
    ax.set_title(title, fontsize=9)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")


fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
draw_boundary(axes[0], np.array([1.0, 1.0]),  0.0, "w=(1,1), b=0\nreference")
draw_boundary(axes[1], np.array([1.0, -1.0]), 0.0, "w=(1,-1), b=0\nweights ROTATE it")
draw_boundary(axes[2], np.array([1.0, 1.0]), -1.5, "w=(1,1), b=-1.5\nbias SLIDES it")
draw_boundary(axes[3], np.array([2.0, 2.0]),  0.0, "w=(2,2), b=0\nsame line, steeper z")
fig.suptitle("One neuron = one straight boundary.  Red arrow = the weight vector w (the normal).", y=1.04)
plt.tight_layout(); plt.show()

print("Panels 1 and 4: doubling w does NOT move the boundary - only the scale of z changes.")
print("That matters later: the unit step throws that scale away, Adaline's linear activation keeps it.")

### 🎛️ Interactive — Boundary explorer

Drag $w_1$, $w_2$ and $b$ and watch the line move. If widgets are unavailable, the cell prints a small
static sweep instead.

In [ ]:
def explore(w1=1.0, w2=1.0, b=0.0):
    """Draw the boundary for one (w1, w2, b) and print its slope-intercept form."""
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    draw_boundary(ax, np.array([w1, w2]), b, f"w=({w1:.1f}, {w2:.1f}), b={b:.1f}")
    plt.show()
    if abs(w2) > 1e-9:
        print(f"boundary:  x2 = {-w1 / w2:+.2f} * x1 {-b / w2:+.2f}")
    else:
        print("boundary is vertical (w2 = 0)")


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             w1=FloatSlider(1.0, min=-3, max=3, step=0.25),
             w2=FloatSlider(1.0, min=-3, max=3, step=0.25),
             b=FloatSlider(0.0, min=-3, max=3, step=0.25))
except ImportError:
    print("[ipywidgets not available - showing a static sweep]")
    for w1, w2, b in [(1.0, 1.0, 0.0), (1.0, -1.0, 0.0), (1.0, 1.0, -1.5)]:
        explore(w1, w2, b)

### ✍️ Exercise 1 — Hand-build gates, then meet the wall

1. Find a weight and a bias for the **NOT** gate (one input: $0 \to 1$, $1 \to 0$). Verify with `neuron`.
2. Find weights and a bias for a gate that fires only for the pattern $(1, 0)$ — "x1 AND NOT x2".
3. Now try **XOR** (fire for $(0,1)$ and $(1,0)$, not for $(0,0)$ and $(1,1)$). Brute-force over
   $w_1, w_2, b \in \{-2, -1.5, \dots, 2\}$ and count how many candidates solve it. What do you find, and
   how does the picture in §1.4 explain it?

*Hint for 3:* `itertools.product` over `np.arange(-2, 2.01, 0.5)`.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) NOT   : w, b such that neuron([[0]], w, b) == 1 and neuron([[1]], w, b) == 0
# 2) (1,0) : x1 AND NOT x2
# 3) XOR   : brute-force search over w1, w2, b  ->  how many solutions?

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import itertools
import numpy as np

# 1) NOT - a single negative weight and a positive bias
w_not, b_not = np.array([-1.0]), 0.5
print("NOT:", [neuron(np.array([[x]]), w_not, b_not)[0] for x in (0, 1)])        # [1, 0]

# 2) x1 AND NOT x2
w_gate, b_gate = np.array([1.0, -1.0]), -0.5
print("x1 AND NOT x2:", [neuron(np.array([[a, b]]), w_gate, b_gate)[0]
                         for a, b in INPUTS])                                    # [0, 0, 1, 0]

# 3) XOR - exhaustive search
target = np.array([0, 1, 1, 0])                        # XOR of INPUTS
grid = np.arange(-2, 2.01, 0.5)
solutions = 0
for w1, w2, b in itertools.product(grid, grid, grid):
    if np.array_equal(neuron(INPUTS, np.array([w1, w2]), b), target):
        solutions += 1
print("XOR solutions found:", solutions, "out of", len(grid) ** 3, "candidates")  # 0
```

**Why zero.** XOR needs $(0,1)$ and $(1,0)$ on one side of the boundary and $(0,0)$, $(1,1)$ on the other.
Those pairs sit on the two *diagonals* of the square, so no straight line separates them — and §1.4 showed
a single neuron can only ever draw a straight line. No learning rule can fix this; the **model** is too
small, not the training. XOR is the standard argument for **hidden layers** ([§7](#s7)), where you solve it.

</details>

---
<a id="s2"></a>
# 2 · The Iris Dataset

[⬆ back to TOC](#toc)

**Iris** (Fisher, 1936) is the *Hello World* of classification: **150 flowers**, 50 from each of three
species, with **four measurements in centimetres** — sepal length, sepal width, petal length, petal width.

Two properties of the labels matter today:

1. The classes are stored as **text** (`"setosa"`, `"versicolor"`, `"virginica"`). Our neuron does
   arithmetic, so they must be encoded as numbers.
2. The classes are **nominal**: mapping them to 0 / 1 / 2 is *arbitrary*. Nothing says versicolor lies
   "between" the other two, and nothing says virginica is "twice" versicolor. A model that consumes that
   integer as a quantity will believe an ordering that does not exist. The fix for the multi-class case is
   **one-hot encoding** — it returns in [§7](#s7).

Today's neuron is **binary**, so we use **setosa vs versicolor** with **two features**, and
$y \in \{0, 1\}$.

### 🧪 Demo — Load and inspect

We load Iris from scikit-learn because it ships with the package and therefore always works offline.
The textbook loads the same 150 rows straight from the UCI archive, which is worth knowing:

```python
import pandas as pd
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
df = pd.read_csv(url, header=None, encoding="utf-8")
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

df = iris.frame.copy()
# The bundled version stores the species as an integer code; put the TEXT label back,
# because that is how the raw data really looks.
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names).astype(str)
df = df.drop(columns=["target"])
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

print("shape:", df.shape)
print()
print(df.head())
print("...")
print(df.tail(3))

In [ ]:
print("dtypes -- note that 'species' is text (object), not a number:")
print(df.dtypes)
print()
print("50 examples per class, perfectly balanced:")
print(df["species"].value_counts())
print()
print("summary statistics (centimetres):")
print(df.describe().round(2))

### 🧪 Demo — Plot the features against the labels

Before any model: **look at the data**. Two questions to answer with your eyes.

1. Which features separate the classes? (Petal measurements will win by a lot.)
2. Is any pair of classes separable by a **straight line**? That is exactly the condition under which the
   perceptron is guaranteed to converge ([§3](#s3)).

In [ ]:
import matplotlib.pyplot as plt

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
SPECIES = ["setosa", "versicolor", "virginica"]
COLORS = {"setosa": "#3b6fb6", "versicolor": "#d1495b", "virginica": "#66a182"}
MARKERS = {"setosa": "o", "versicolor": "s", "virginica": "^"}

fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, feature in zip(axes.ravel(), FEATURES):
    for name in SPECIES:
        ax.hist(df.loc[df["species"] == name, feature], bins=12, alpha=0.6,
                label=name, color=COLORS[name])
    ax.set_xlabel(f"{feature} [cm]")
    ax.set_ylabel("count")
axes[0, 0].legend(fontsize=8)
fig.suptitle("One histogram per feature. Petal measurements separate the classes; sepal width does not.")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for name in SPECIES:
    subset = df[df["species"] == name]
    axes[0].scatter(subset["sepal_length"], subset["sepal_width"],
                    c=COLORS[name], marker=MARKERS[name], edgecolor="k",
                    linewidth=0.4, s=45, label=name)
    axes[1].scatter(subset["petal_length"], subset["petal_width"],
                    c=COLORS[name], marker=MARKERS[name], edgecolor="k",
                    linewidth=0.4, s=45, label=name)

axes[0].set(xlabel="sepal length [cm]", ylabel="sepal width [cm]", title="sepal features: overlapping")
axes[1].set(xlabel="petal length [cm]", ylabel="petal width [cm]", title="petal features: nearly separable")
axes[1].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

print("Read the right-hand panel carefully:")
print("  setosa vs the rest      -> a straight line separates them with room to spare")
print("  versicolor vs virginica -> they touch. No straight line gets 100%. Remember this for section 3.")

### 🧪 Demo — Fixing the notation

This notation is used for the rest of the course, so it is worth spelling out once:

$$X \in \mathbb{R}^{150 \times 4}, \qquad
\mathbf{x}^{(i)} = \text{row } i = \text{one example}, \qquad
x_j = \text{column } j = \text{one feature}$$

* superscript $(i)$ indexes the **example**, subscript $j$ the **feature**
* $n$ = number of examples, $m$ = number of features
* $y^{(i)}$ is the true label, $\hat{y}^{(i)}$ the prediction

In [ ]:
X_all = df[FEATURES].to_numpy()                 # (150, 4)
y_text = df["species"].to_numpy()               # (150,) of strings

n, m = X_all.shape
print(f"X_all.shape = {X_all.shape}   ->  n = {n} examples, m = {m} features")
print()
print("x^(0)  (row 0, one FLOWER, all four measurements):", X_all[0])
print("x_2    (column 2, one FEATURE across all flowers):", X_all[:5, 2], "...")
print()
print("x^(0)_2  = feature 2 of example 0 =", X_all[0, 2])
print("y^(0)    =", y_text[0])

# ---- the binary, two-feature problem for sections 3-6 -------------------------
mask = np.isin(y_text, ["setosa", "versicolor"])                 # first 100 rows
X = X_all[mask][:, [0, 2]]                                       # sepal length, petal length
y = np.where(y_text[mask] == "setosa", 0, 1).astype(float)       # setosa -> 0, versicolor -> 1

print()
print("binary problem:", X.shape, "labels:", np.unique(y), "-> counts:", np.bincount(y.astype(int)))
print("feature ranges  :", X.min(axis=0), "to", X.max(axis=0))

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.2))
ax.scatter(X[y == 0, 0], X[y == 0, 1], c="#3b6fb6", marker="o", edgecolor="k",
           linewidth=0.4, s=50, label="setosa (y=0)")
ax.scatter(X[y == 1, 0], X[y == 1, 1], c="#d1495b", marker="s", edgecolor="k",
           linewidth=0.4, s=50, label="versicolor (y=1)")
ax.plot([4.0, 7.2], [3.1, 1.0], "k--", linewidth=1.2, label="one separating line (by eye)")
ax.set(xlabel="sepal length [cm]", ylabel="petal length [cm]",
       title="Linearly separable: infinitely many lines work")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("The perceptron's job in section 3 is to FIND one of those lines from the labels alone.")

### ✍️ Exercise 2 — Look before you model

1. Build `X_hard`, `y_hard` for **versicolor vs virginica** using the same two features
   (`sepal_length`, `petal_length`). Plot them the way the cell above does.
2. Can you draw a straight line that separates them perfectly? Try — and count the points you get wrong.
3. Repeat with `petal_length` and `petal_width` instead. Does the *choice of features* change your answer?
4. Compute the per-class mean of every feature (`df.groupby("species").mean()`) and say which single
   feature separates setosa from the other two best.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) X_hard, y_hard for versicolor vs virginica, then plot
# 2) try a straight line by eye and count the errors
# 3) repeat with petal_length + petal_width
# 4) df.groupby("species").mean()

<details>
<summary>✅ <b>Show solution</b></summary>

```python
mask_hard = np.isin(y_text, ["versicolor", "virginica"])
X_hard = X_all[mask_hard][:, [0, 2]]                                    # sepal + petal length
y_hard = np.where(y_text[mask_hard] == "versicolor", 0, 1).astype(float)

X_petal = X_all[mask_hard][:, [2, 3]]                                   # petal length + width

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, xlabel, ylabel, title in [
        (axes[0], X_hard, "sepal length", "petal length", "sepal+petal length: overlapping"),
        (axes[1], X_petal, "petal length", "petal width", "petal length+width: almost separable")]:
    ax.scatter(data[y_hard == 0, 0], data[y_hard == 0, 1], c="#d1495b", marker="s",
               edgecolor="k", linewidth=0.4, label="versicolor")
    ax.scatter(data[y_hard == 1, 0], data[y_hard == 1, 1], c="#66a182", marker="^",
               edgecolor="k", linewidth=0.4, label="virginica")
    ax.set(xlabel=f"{xlabel} [cm]", ylabel=f"{ylabel} [cm]", title=title)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# 2) a hand-picked line, and how many points it gets wrong
w_eye, b_eye = np.array([0.0, 1.0]), -4.9        # "petal length >= 4.9 -> virginica"
errors = int((unit_step(X_hard @ w_eye + b_eye) != y_hard).sum())
print(f"hand-picked line gets {errors} of {len(y_hard)} wrong")     # a handful, never 0

# 4) class means
print(df.groupby("species")[FEATURES].mean().round(2))
```

**What to take away.** Versicolor and virginica **overlap** in every feature pair — the best straight line
still misses a few points. That is not a training failure, it is the data. Keep it in mind: the perceptron
is only *guaranteed* to converge when the classes are linearly separable, and on this pair it never will.
The class means also show why petals win: setosa's mean petal length is ~1.46 cm against ~4.26 and
~5.55 cm — a gap no sepal measurement comes close to.

</details>

---
<a id="s3"></a>
# 3 · The Perceptron Learning Rule (1957)

[⬆ back to TOC](#toc)

Rosenblatt's contribution is not the neuron — it is the **rule that finds the weights**. For one example
$i$ at a time:

$$\boxed{\;\Delta w_j = \eta\,\bigl(y^{(i)} - \hat{y}^{(i)}\bigr)\,x_j^{(i)}, \qquad
\Delta b = \eta\,\bigl(y^{(i)} - \hat{y}^{(i)}\bigr)\;}$$

where $\eta$ is the **learning rate** and $\hat{y}^{(i)} = \sigma(z^{(i)})$ is the **thresholded**
prediction. Four consequences follow, and all four are visible in the next cell:

| | Consequence |
|:--|:--|
| **1** | **Correct prediction → no update.** $y - \hat{y} = 0$ makes both $\Delta$ terms zero. The rule learns *only from mistakes*, which is also why it stops once the data is separated. |
| **2** | **Wrong prediction → the boundary moves the right way**, in both error directions ($+1$ and $-1$). |
| **3** | **The size of the correction scales with the feature.** $x_j = 2.0$ pulls its weight twice as hard as $x_j = 1.0$ — so a feature measured on a larger scale drags the weights harder. This is the motivation for **feature scaling** ([§5](#s5)). |
| **4** | **There is no $x$ in the bias update.** The bias has no input to scale it, so it always moves by $\eta\,(y - \hat{y})$. |

### 🧪 Demo — One update, by hand

`perceptron_update` does exactly what the boxed formula says, and prints every intermediate number.
Read the four cases against the four rows of the table above.

In [ ]:
import numpy as np

def perceptron_update(x, y_true, w, b, eta=0.1, verbose=True):
    """
    Apply the perceptron rule for ONE example and report every intermediate value.

    Args:
        x: One example, shape (m,).
        y_true: Its true label, 0 or 1.
        w: Current weights, shape (m,).
        b: Current bias, a scalar.
        eta: Learning rate.
        verbose: Print the arithmetic.

    Returns:
        (w_new, b_new) after the update.
    """
    z = float(x @ w + b)
    y_hat = 1.0 if z >= 0.0 else 0.0
    error = y_true - y_hat                       # in {-1, 0, +1} for a binary perceptron

    delta_w = eta * error * x                    # scales with EACH feature
    delta_b = eta * error                        # no x here at all

    if verbose:
        print(f"  x = {x}, y = {y_true:.0f}")
        print(f"  z = w.x + b = {z:+.3f}  ->  y_hat = {y_hat:.0f}   error = y - y_hat = {error:+.0f}")
        print(f"  delta_w = eta * error * x = {eta} * {error:+.0f} * {x} = {delta_w}")
        print(f"  delta_b = eta * error     = {eta} * {error:+.0f}          = {delta_b:+.3f}")
        print(f"  w: {w} -> {w + delta_w}     b: {b:+.3f} -> {b + delta_b:+.3f}")
    return w + delta_w, b + delta_b


w0, b0 = np.array([0.2, -0.3]), 0.0

print("CASE 1 - prediction is already CORRECT (consequence 1: nothing moves)")
perceptron_update(np.array([1.0, 2.0]), y_true=0.0, w=w0, b=b0)

print("\nCASE 2a - WRONG, we said 0 but the truth is 1  (error = +1: weights grow)")
perceptron_update(np.array([1.0, 1.0]), y_true=1.0, w=np.array([-0.5, -0.5]), b=0.0)

print("\nCASE 2b - WRONG the other way, we said 1 but the truth is 0  (error = -1: weights shrink)")
perceptron_update(np.array([1.0, 1.0]), y_true=0.0, w=np.array([0.5, 0.5]), b=0.0)

print("\nCASE 3 - the SAME error, but feature 1 is on a bigger scale (consequence 3 + 4)")
perceptron_update(np.array([1.5, 1.0]), y_true=1.0, w=np.array([-0.5, -0.5]), b=0.0)
perceptron_update(np.array([2.0, 1.0]), y_true=1.0, w=np.array([-0.5, -0.5]), b=0.0)
print("  ^ delta_w[0] grew from 0.15 to 0.20 with x[0], while delta_b stayed at +0.100.")

### 🧪 Demo — Why the update moves the boundary the *right* way

Consequence 2 deserves a picture. Take one misclassified point, apply one update, and watch the line
swing towards it. The update is $\Delta\mathbf{w} = \eta\,(y-\hat{y})\,\mathbf{x}$, which is a step
*along $\pm\mathbf{x}$ itself* — so the net input for that very point necessarily moves in the direction
that fixes it:

$$z_{\text{new}} = (\mathbf{w} + \eta\,e\,\mathbf{x})^\top\mathbf{x} + b + \eta e
= z_{\text{old}} + \eta\,e\,(\|\mathbf{x}\|^2 + 1)$$

With $e = +1$ that is strictly larger; with $e = -1$ strictly smaller. Enough repetitions and the point
crosses the boundary.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_bad = np.array([1.2, 0.8])          # a point whose true label is 1 ...
y_bad = 1.0
w_before, b_before = np.array([-0.6, -0.2]), 0.1     # ... but this boundary says 0

w_after, b_after = perceptron_update(x_bad, y_bad, w_before, b_before, eta=0.4, verbose=False)

z_before = float(x_bad @ w_before + b_before)
z_after = float(x_bad @ w_after + b_after)
print(f"z before = {z_before:+.3f} (predicts 0 - wrong)")
print(f"z after  = {z_after:+.3f} (predicts {1 if z_after >= 0 else 0} - fixed in a single step)")
print(f"predicted increase eta*(|x|^2 + 1) = {0.4 * (x_bad @ x_bad + 1):+.3f}"
      f"   actual = {z_after - z_before:+.3f}")

fig, ax = plt.subplots(figsize=(5.2, 4.4))
line_x = np.linspace(-1.5, 2.5, 50)
for w_, b_, style, label in [(w_before, b_before, "r--", "before update"),
                             (w_after, b_after, "g-", "after ONE update")]:
    ax.plot(line_x, -(w_[0] * line_x + b_) / w_[1], style, label=label)
ax.scatter(*x_bad, s=160, c="#d1495b", marker="*", edgecolor="k", zorder=5,
           label="misclassified point (y=1)")
ax.set(xlim=(-1.5, 2.5), ylim=(-1.5, 2.5), xlabel="x1", ylabel="x2",
       title="One update swings the boundary past the point")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3.1 The full algorithm

Assembling the loop:

1. Initialise $\mathbf{w}$ and $b$ (small random numbers).
2. For each **epoch**, for each example $\mathbf{x}^{(i)}$:
   * compute the net input $z$, threshold it into $\hat{y}$
   * compare with $y^{(i)}$ and update $\mathbf{w}, b$ from the error
3. Count the misclassifications per epoch. **Zero errors ⇒ nothing can change any more ⇒ converged.**

In [ ]:
loop_diagram = """
flowchart LR
    X[("inputs x")] --> Z["net input<br/>z = wᵀx + b"]
    W[("weights w, b")] --> Z
    Z --> T["unit step"]
    T --> YH(("y-hat"))
    YH --> C{"compare<br/>with y"}
    Y[("true label y")] --> C
    C -->|"error = y - y-hat"| U["update<br/>dw = eta*e*x<br/>db = eta*e"]
    U -->|"only if error != 0"| W
"""
render_mermaid(loop_diagram)

In [ ]:
import numpy as np

class Perceptron:
    """
    Rosenblatt's perceptron: a single neuron trained with the perceptron rule.

    Args:
        eta: Learning rate, in (0, 1].
        n_iter: Number of passes (epochs) over the training set.
        random_state: Seed for the initial weights.

    Attributes:
        w_: Weights after fitting, shape (m,).
        b_: Bias after fitting, a scalar.
        errors_: Number of misclassifications in each epoch.
    """

    def __init__(self, eta=0.01, n_iter=50, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def net_input(self, X):
        """z = X w + b."""
        return X @ self.w_ + self.b_

    def activation(self, z):
        """The unit step - this is what the perceptron measures its error from."""
        return np.where(z >= 0.0, 1.0, 0.0)

    def predict(self, X):
        """Class labels in {0, 1}."""
        return self.activation(self.net_input(X))

    def fit(self, X, y):
        """Learn from X (n, m) and y (n,) with labels in {0, 1}."""
        rng = np.random.default_rng(self.random_state)
        self.w_ = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = 0.0
        self.errors_ = []

        for _ in range(self.n_iter):
            errors = 0
            for xi, target in zip(X, y):                  # ONE EXAMPLE AT A TIME
                error = target - self.predict(xi.reshape(1, -1))[0]
                self.w_ += self.eta * error * xi          # delta_w = eta * e * x
                self.b_ += self.eta * error               # delta_b = eta * e
                errors += int(error != 0.0)               # count mistakes, not their size
            self.errors_.append(errors)
        return self


perceptron = Perceptron(eta=0.01, n_iter=20).fit(X, y)

print("misclassifications per epoch:", perceptron.errors_)
print()
print(f"learned w = {perceptron.w_.round(3)}, b = {perceptron.b_:+.3f}")
print(f"training accuracy = {(perceptron.predict(X) == y).mean():.1%}")
first_zero = perceptron.errors_.index(0) + 1
print(f"converged in epoch {first_zero}; every later epoch is a no-op (consequence 1).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(range(1, len(perceptron.errors_) + 1), perceptron.errors_, marker="o")
axes[0].set(xlabel="epoch", ylabel="misclassifications", title="Errors go to zero, then stay there")
axes[0].axhline(0, color="grey", linewidth=0.8)

plot_decision_regions(X, y.astype(int), perceptron.predict, ax=axes[1],
                      xlabel="sepal length [cm]", ylabel="petal length [cm]",
                      title="The line the perceptron found",
                      class_names=["setosa", "versicolor"])
plt.tight_layout(); plt.show()

## 3.2 The catch: convergence is only guaranteed if the classes are linearly separable

The perceptron stops when it makes no mistakes. If **no** line makes zero mistakes, **it never stops
updating** — some example is misclassified in every single epoch, so $\mathbf{w}$ and $b$ are still being
nudged when the epoch budget runs out. Watch it on versicolor vs virginica, the pair you found overlapping
in Exercise 2.

In [ ]:
mask_hard = np.isin(y_text, ["versicolor", "virginica"])
X_hard = X_all[mask_hard][:, [0, 2]]
y_hard = np.where(y_text[mask_hard] == "versicolor", 0, 1).astype(float)

hard_perceptron = Perceptron(eta=0.01, n_iter=60).fit(X_hard, y_hard)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(range(1, 61), hard_perceptron.errors_, marker=".", color="#d1495b")
axes[0].set(xlabel="epoch", ylabel="misclassifications",
            title="Not separable: the error never reaches 0")
axes[0].set_ylim(0, max(hard_perceptron.errors_) + 1)
axes[0].axhline(0, color="grey", linewidth=0.8)

plot_decision_regions(X_hard, y_hard.astype(int), hard_perceptron.predict, ax=axes[1],
                      xlabel="sepal length [cm]", ylabel="petal length [cm]",
                      title="... and the boundary is wherever epoch 60 left it",
                      class_names=["versicolor", "virginica"])
plt.tight_layout(); plt.show()

print("errors per epoch, first 10 :", hard_perceptron.errors_[:10])
print("errors per epoch, last 10  :", hard_perceptron.errors_[-10:])
print("did it EVER reach zero?    :", 0 in hard_perceptron.errors_)
print()

# The error count sits still, but the parameters do not: those 2 mistakes cause 2 updates
# in every single epoch, forever.
longer = Perceptron(eta=0.01, n_iter=90).fit(X_hard, y_hard)
print(f"weights after 60 epochs : w = {hard_perceptron.w_.round(4)}, b = {hard_perceptron.b_:+.3f}")
print(f"weights after 90 epochs : w = {longer.w_.round(4)}, b = {longer.b_:+.3f}")
print("still moving             :", not np.allclose(hard_perceptron.w_, longer.w_))
print()
print("Two problems, and they are the reason section 4 exists:")
print("  1. There is no LOSS here - 'number of mistakes' is what we count, not what we minimise,")
print("     so nothing tells us that epoch 60 is better or worse than epoch 59.")
print("  2. Training never converges. It stops only by running out of epochs, keeping whatever")
print("     weights happened to be current at that arbitrary moment.")

### ✍️ Exercise 3 — Push the perceptron around

1. Train on the separable problem (`X`, `y`) with `eta` in `[0.0001, 0.01, 0.1, 1.0]`, 20 epochs each.
   Plot the four error curves in one figure. Does a bigger `eta` always converge sooner?
2. Explain the result of 1 using consequences 1–4. (*Hint:* what does the **initial** $\mathbf{w}$ do
   when $\eta$ is 100× larger, and does scaling $\mathbf{w}$ change the boundary at all? Look back at
   panels 1 and 4 of §1.4.)
3. Swap in the other feature pair `[2, 3]` (petal length, petal width) for setosa vs versicolor. Does it
   converge faster? Why?
4. Take the trained perceptron from the separable problem, multiply `w_` and `b_` by 10, and re-measure the
   accuracy. Explain what you see.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) sweep eta over [0.0001, 0.01, 0.1, 1.0] and plot the four error curves
# 2) written answer, in a comment
# 3) X_petal = X_all[mask][:, [2, 3]] ; retrain
# 4) scale w_ and b_ by 10, re-check the accuracy

<details>
<summary>✅ <b>Show solution</b></summary>

```python
# 1) learning-rate sweep
fig, ax = plt.subplots()
for eta in [0.0001, 0.01, 0.1, 1.0]:
    model = Perceptron(eta=eta, n_iter=20).fit(X, y)
    epochs_to_converge = (model.errors_.index(0) + 1) if 0 in model.errors_ else None
    ax.plot(range(1, 21), model.errors_, marker="o", markersize=3,
            label=f"eta={eta} (converged: {epochs_to_converge})")
ax.set(xlabel="epoch", ylabel="misclassifications", title="All four converge on separable data")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# 3) petal features
X_petal = X_all[mask][:, [2, 3]]
petal_model = Perceptron(eta=0.01, n_iter=20).fit(X_petal, y)
print("petal features, errors per epoch:", petal_model.errors_)

# 4) scaling the learned parameters
scaled = Perceptron().fit(X, y)
scaled.w_, scaled.b_ = scaled.w_ * 10, scaled.b_ * 10
print("accuracy after scaling w and b by 10:", (scaled.predict(X) == y).mean())
```

**2 — why `eta` barely matters here.** On separable data every `eta` converges; only the *number of
epochs* wobbles. The reason is that the boundary depends on the **direction** of $\mathbf{w}$ and on the
ratio $b / \|\mathbf{w}\|$, not on the magnitude of $\mathbf{w}$ (§1.4, panels 1 and 4). Multiplying
$\eta$ by 100 multiplies every update by 100, which mostly just makes the tiny random initial weights
irrelevant faster — it does not change the *set* of lines the rule can reach. `eta` becomes genuinely
important in §4, where the loss has a curved surface and the step size can overshoot the minimum.

**4 — the accuracy is unchanged.** $10z$ has the same sign as $z$, and the unit step only looks at the
sign. The perceptron literally cannot tell a confident prediction from a marginal one — it throws that
information away. Adaline keeps it, and that is the whole difference.

**3 — petals converge sooner** because the gap between the two classes is much wider relative to the
feature spread, so almost any line drawn early is already correct.

</details>

---
<a id="s4"></a>
# 4 · Adaline and Gradient Descent (1960)

[⬆ back to TOC](#toc)

## 4.1 The one change

**ADALINE** (ADAptive LInear NEuron, Widrow & Hoff 1960) is the same network with **one edge moved**: the
error is measured from the **continuous** activation $\sigma(z) = z$ instead of from the **thresholded**
label. Everything else in this section follows from that single change.

| | Perceptron (1957) | Adaline (1960) |
|:--|:--|:--|
| Learning activation | unit step, $\sigma(z) \in \{0, 1\}$ | **linear**, $\sigma(z) = z$ |
| Error measured from | thresholded $\hat{y}$ | continuous $\sigma(z)$ |
| Loss function | **none** — it minimises nothing | **MSE**: differentiable *and* convex |
| Weight update | hand-written correction, one example, only on a mistake | **gradient descent** over all $n$, every step |
| Converges | only if linearly separable | always, to the single MSE minimum |
| Prediction | unit step | unit step (**at prediction time only**) |

> 🔑 **Why the perceptron has no loss to descend.** The unit step is flat everywhere except at $z=0$,
> where it is not differentiable at all. Its derivative is therefore **zero wherever it exists** — a
> gradient of zero tells you nothing about which way to move. That is not a detail of the perceptron; it
> is the reason the choice of activation is the *first* thing that matters in a deep network ([§10](#s10)).

In [ ]:
comparison_diagram = """
flowchart LR
    subgraph P["Perceptron: error from the THRESHOLDED output"]
        px[("x")] --> pz["z = wᵀx + b"] --> pstep["unit step"] --> pe{"y - y-hat"}
        pe -->|update| pz
    end
    subgraph A["Adaline: error from the CONTINUOUS output"]
        ax[("x")] --> az["z = wᵀx + b"] --> alin["linear: sigma(z) = z"] --> ae{"y - sigma(z)"}
        ae -->|"gradient of MSE"| az
        alin --> astep["unit step<br/>(prediction only)"]
    end
"""
render_mermaid(comparison_diagram)

## 4.2 The loss, and why it can be descended

$$L(\mathbf{w}, b) \;=\; \frac{1}{2n}\sum_{i=1}^{n}\Bigl(y^{(i)} - \sigma\bigl(z^{(i)}\bigr)\Bigr)^{2},
\qquad \sigma(z) = z$$

* The $\tfrac{1}{2}$ is **pure convenience** — it cancels the 2 that differentiating the square produces.
* Because the activation is **linear**, the loss is **differentiable** everywhere and **convex** — one
  global minimum, no local traps. Gradient descent on it cannot get stuck.
* Contrast this with §8 onwards: as soon as we add a non-linear hidden layer the loss stops being convex,
  and we need backpropagation to get the derivatives at all.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# What each model measures its error from, for a single example with y = 1.
z_values = np.linspace(-3, 3, 400)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(z_values, np.where(z_values >= 0, 1.0, 0.0), color="#3b6fb6")
axes[0].set(title="Perceptron activation: unit step", xlabel="z", ylabel="sigma(z)")

axes[1].plot(z_values, np.zeros_like(z_values), color="#d1495b")
axes[1].set(title="its derivative: ZERO everywhere\n(nothing to descend)", xlabel="z",
            ylabel="d sigma / dz", ylim=(-1, 1))

axes[2].plot(z_values, 0.5 * (1.0 - z_values) ** 2, color="#66a182")
axes[2].plot(z_values, -(1.0 - z_values), "--", color="#c98b2e", label="dL/dz")
axes[2].axvline(1.0, color="grey", linewidth=0.8)
axes[2].set(title="Adaline MSE for y=1: a bowl\nwith a usable slope", xlabel="z", ylabel="loss")
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Middle panel: a zero gradient carries no information about which way to move.")
print("Right panel : the slope is negative left of the target and positive right of it -> it points home.")

## 4.3 The derivative, step by step

This is the derivation you should be able to reproduce on paper. Differentiate the loss with respect to
one weight $w_j$:

$$\frac{\partial L}{\partial w_j}
= \frac{\partial}{\partial w_j}\;\frac{1}{2n}\sum_i \bigl(y^{(i)} - \sigma(z^{(i)})\bigr)^2
\;\overset{\text{(1)}}{=}\; \frac{1}{2n}\sum_i 2\bigl(y^{(i)} - \sigma(z^{(i)})\bigr)
  \frac{\partial}{\partial w_j}\bigl(y^{(i)} - \sigma(z^{(i)})\bigr)$$

$$\overset{\text{(2)}}{=}\;\frac{1}{n}\sum_i \bigl(y^{(i)} - \sigma(z^{(i)})\bigr)
  \frac{\partial}{\partial w_j}\Bigl(y^{(i)} - \textstyle\sum_k w_k x_k^{(i)} - b\Bigr)
\;\overset{\text{(3)}}{=}\; \boxed{\,-\frac{1}{n}\sum_i \bigl(y^{(i)} - \sigma(z^{(i)})\bigr)\,x_j^{(i)}\,}$$

1. chain rule on the square; the $\tfrac12$ eats the 2
2. substitute $\sigma(z) = z = \sum_k w_k x_k + b$ — this is where linearity is used
3. only the $k = j$ term survives, with $\partial(-w_jx_j)/\partial w_j = -x_j$

The bias is **identical except that the inner derivative is $-1$** instead of $-x_j$ (the bias has no
input to scale it — the same fact as consequence 4 in §3):

$$\frac{\partial L}{\partial b} = -\frac{1}{n}\sum_i \bigl(y^{(i)} - \sigma(z^{(i)})\bigr)$$

**Gradient descent** then steps *opposite* the gradient, by a distance set by the learning rate $\eta$:

$$\mathbf{w} := \mathbf{w} - \eta\,\nabla_{\mathbf{w}}L
= \mathbf{w} + \frac{\eta}{n}\sum_i \bigl(y^{(i)} - \sigma(z^{(i)})\bigr)\mathbf{x}^{(i)},
\qquad b := b + \frac{\eta}{n}\sum_i \bigl(y^{(i)} - \sigma(z^{(i)})\bigr)$$

Note the sign flip: subtracting a negative gradient means we **add** the error-weighted inputs. In NumPy
the whole sum over $i$ is one matrix product, `X.T @ errors`.

### 🧪 Demo — Never trust a gradient you have not checked

A wrong derivative is the classic silent bug: the model still runs, the loss still moves, it just learns
badly. The cure is a **numerical gradient check**. The definition of a derivative gives us a second,
independent estimate — a central finite difference:

$$\frac{\partial L}{\partial w_j} \approx \frac{L(w_j + \varepsilon) - L(w_j - \varepsilon)}{2\varepsilon}$$

It is far too slow to train with (one loss evaluation per parameter per step), but it is perfect for
*verifying* the analytic formula on a small example. **Do this every single time you write a gradient by
hand** — including in §8, where we check a whole network this way.

In [ ]:
import numpy as np

def mse_loss(X, y, w, b):
    """Adaline's loss: L = 1/(2n) * sum (y - z)^2, with z = Xw + b."""
    errors = y - (X @ w + b)
    return float((errors ** 2).sum() / (2 * len(y)))


def mse_grad_analytic(X, y, w, b):
    """The boxed formulas from 4.3, vectorised."""
    errors = y - (X @ w + b)                 # (n,)
    grad_w = -(X.T @ errors) / len(y)        # (m,)   the sum over i IS this matrix product
    grad_b = -errors.sum() / len(y)          # scalar, no x anywhere
    return grad_w, grad_b


def mse_grad_numeric(X, y, w, b, eps=1e-6):
    """The same gradient by central finite differences - slow, but assumption-free."""
    grad_w = np.zeros_like(w)
    for j in range(len(w)):
        step = np.zeros_like(w)
        step[j] = eps
        grad_w[j] = (mse_loss(X, y, w + step, b) - mse_loss(X, y, w - step, b)) / (2 * eps)
    grad_b = (mse_loss(X, y, w, b + eps) - mse_loss(X, y, w, b - eps)) / (2 * eps)
    return grad_w, grad_b


# A small random problem - a gradient check should always be run on something tiny.
X_check = RNG.normal(size=(7, 3))
y_check = RNG.integers(0, 2, size=7).astype(float)
w_check, b_check = RNG.normal(scale=0.5, size=3), 0.3

gw_a, gb_a = mse_grad_analytic(X_check, y_check, w_check, b_check)
gw_n, gb_n = mse_grad_numeric(X_check, y_check, w_check, b_check)

print("analytic dL/dw :", gw_a.round(8))
print("numeric  dL/dw :", gw_n.round(8))
print("analytic dL/db : %+.8f" % gb_a)
print("numeric  dL/db : %+.8f" % gb_n)

relative = np.abs(gw_a - gw_n).max() / max(np.abs(gw_a).max(), 1e-12)
print(f"\nmax relative difference: {relative:.2e}   ->", "PASS" if relative < 1e-5 else "FAIL")
print("Anything below about 1e-5 is finite-difference round-off, not a bug in the formula.")

# And this is what a WRONG gradient looks like - a forgotten minus sign:
gw_wrong = (X_check.T @ (y_check - (X_check @ w_check + b_check))) / len(y_check)
print("\nsign-flipped 'gradient' vs numeric -> relative difference:",
      f"{np.abs(gw_wrong - gw_n).max() / np.abs(gw_n).max():.2e}  <- FAIL, caught immediately")

## 4.4 The learning rate: how far to step

The gradient gives the **direction**; $\eta$ gives the **distance**. Take the simplest possible bowl,
$L(w) = \tfrac12 w^2$, whose gradient is $w$. One step is then

$$w := w - \eta\,w = (1 - \eta)\,w$$

so the behaviour is decided entirely by $|1 - \eta|$:

| $\eta$ | $|1-\eta|$ | What happens |
|:--|:--|:--|
| 0.1 | 0.9 | creeps in, monotonically — safe but slow |
| 1.0 | 0.0 | lands on the minimum in a single step (only because this bowl is that simple) |
| 1.5 | 0.5 | overshoots, alternates sides, still converges |
| **2.0** | **1.0** | **oscillates forever** — every step overshoots by exactly what it started away from |
| 2.5 | 1.5 | **diverges** — the loss *climbs* |

The threshold is not a universal number: it depends on the curvature of the loss. What is universal is the
shape of the failure — **too large an $\eta$ overshoots the minimum by more than it started away from it.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def descend_1d(eta, w0=2.0, steps=12):
    """Gradient descent on L(w) = 0.5 w^2, whose gradient is exactly w."""
    path = [w0]
    w = w0
    for _ in range(steps):
        w = w - eta * w                     # w := w - eta * dL/dw
        path.append(w)
    return np.array(path)


w_axis = np.linspace(-3.2, 3.2, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for eta, color in [(0.1, "#3b6fb6"), (1.0, "#66a182"), (1.5, "#c98b2e"),
                   (2.0, "#d1495b"), (2.5, "#7d5ba6")]:
    path = descend_1d(eta)
    axes[0].plot(path, 0.5 * path ** 2, "o-", color=color, markersize=4,
                 alpha=0.85, label=f"eta={eta}")
    axes[1].plot(0.5 * path ** 2, "o-", color=color, markersize=4, label=f"eta={eta}")

axes[0].plot(w_axis, 0.5 * w_axis ** 2, "k-", linewidth=1, alpha=0.4)
axes[0].set(xlabel="w", ylabel="L(w) = 0.5 w²", title="Steps on the bowl", ylim=(-0.5, 6))
axes[0].legend(fontsize=8)
axes[1].set(xlabel="step", ylabel="loss", title="Loss per step (log scale)", yscale="log")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for eta in [0.1, 1.0, 1.5, 2.0, 2.5]:
    path = descend_1d(eta)
    print(f"eta={eta:<4} final w = {path[-1]:+10.4f}   final loss = {0.5 * path[-1] ** 2:12.6f}")

## 4.5 Adaline in code

Compare `fit` with the perceptron's. There are exactly two differences, and both come from the one edge we
moved:

1. the error is taken from `net_input` (linear) instead of from `predict` (thresholded)
2. the update uses the **whole training set at once** — so there is *one indentation level fewer*, no inner
   loop over examples

In [ ]:
import numpy as np

class AdalineGD:
    """
    Adaline trained with FULL-BATCH gradient descent on the MSE loss.

    Args:
        eta: Learning rate.
        n_iter: Number of passes over the training set. One pass = ONE weight update.
        random_state: Seed for the initial weights.

    Attributes:
        w_: Weights after fitting, shape (m,).
        b_: Bias after fitting, a scalar.
        losses_: Mean squared error (times 1/2) after each epoch.
    """

    def __init__(self, eta=0.01, n_iter=20, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def net_input(self, X):
        """z = X w + b."""
        return X @ self.w_ + self.b_

    def activation(self, z):
        """The LINEAR activation. Identity - present only to mark where it lives."""
        return z

    def predict(self, X):
        """Class labels in {0, 1} - the unit step is used at prediction time only."""
        return np.where(self.activation(self.net_input(X)) >= 0.5, 1.0, 0.0)

    def fit(self, X, y):
        """Learn from X (n, m) and y (n,)."""
        rng = np.random.default_rng(self.random_state)
        self.w_ = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = 0.0
        self.losses_ = []
        n = X.shape[0]

        for _ in range(self.n_iter):
            output = self.activation(self.net_input(X))     # (n,) CONTINUOUS, not thresholded
            errors = y - output                             # (n,)

            # w := w - eta * dL/dw   with   dL/dw = -(1/n) X^T errors
            self.w_ += self.eta * (X.T @ errors) / n
            self.b_ += self.eta * errors.sum() / n

            self.losses_.append((errors ** 2).sum() / (2 * n))
        return self


# Sanity check: does the class agree with the gradient we verified above?
model = AdalineGD(eta=0.01, n_iter=1).fit(X_check, y_check)
print("class runs, loss after 1 epoch:", round(model.losses_[0], 6))
print("threshold at 0.5 for prediction, because the targets are 0 and 1")

### 🧪 Demo — Two learning rates on the *raw* Iris features

Now train on the unscaled data of §2. The features live on very different ranges
(sepal length ≈ 4.3–7.0 cm, petal length ≈ 1.0–5.1 cm) and the labels are 0/1.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

big = AdalineGD(eta=0.05, n_iter=15).fit(X, y)
axes[0].plot(range(1, 16), big.losses_, marker="o", color="#d1495b")
axes[0].set(xlabel="epoch", ylabel="mean squared error / 2", yscale="log",
            title="eta = 0.05 -> DIVERGES (note the log axis)")

small = AdalineGD(eta=0.0001, n_iter=15).fit(X, y)
axes[1].plot(range(1, 16), small.losses_, marker="o", color="#3b6fb6")
axes[1].set(xlabel="epoch", ylabel="mean squared error / 2",
            title="eta = 0.0001 -> converges, slowly")
plt.tight_layout(); plt.show()

print(f"eta=0.05   : loss went {big.losses_[0]:.3f} -> {big.losses_[-1]:.3e}  (climbing)")
print(f"eta=0.0001 : loss went {small.losses_[0]:.3f} -> {small.losses_[-1]:.3f}  (falling)")
print(f"eta=0.0001 : training accuracy {(small.predict(X) == y).mean():.1%} - still chance level "
      f"after 15 epochs")
print()
print("The usable window here is narrow: eta=0.03 works, eta=0.05 explodes, and anything")
print("small enough to be safe barely moves. Section 5 widens that window enormously,")
print("without touching eta at all.")

### ✍️ Exercise 4 — Own the gradient

1. **Derive and check the gradient of a different loss.** For the *mean absolute error*
   $L = \frac{1}{n}\sum_i |y^{(i)} - z^{(i)}|$, the derivative is
   $\partial L/\partial w_j = -\frac{1}{n}\sum_i \operatorname{sign}(y^{(i)} - z^{(i)})\,x_j^{(i)}$.
   Implement `mae_loss` and `mae_grad_analytic`, then verify them against `mse_grad_numeric`'s finite
   differences (write a generic numeric checker that takes a loss function).
2. **Break a gradient on purpose.** In `mse_grad_analytic`, divide by `len(y) ** 2` instead of `len(y)`.
   Does the gradient check catch it? Does the training loss still go down? (This is why you check.)
3. **Find the divergence threshold empirically.** For the raw `X`, `y`, bisect `eta` between 0.0001 and
   0.5 to find the largest value whose loss still decreases over 20 epochs. (Check first that the low end
   converges and the high end does not — a bisection whose bracket is wrong just returns its own bound.)
4. On the 1-D bowl, verify the table in §4.4: for each `eta` in `[0.1, 1.0, 1.5, 2.0, 2.5]`, print
   $|1-\eta|$ next to `abs(path[-1] / path[0]) ** (1/12)`. Do they match?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) mae_loss, mae_grad_analytic, and a generic numeric_grad(loss_fn, ...)
# 2) break the /n and see whether the check catches it
# 3) bisect eta to find the divergence threshold
# 4) verify |1 - eta| against the observed contraction rate

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import numpy as np

# ---- 1) a generic numeric checker, reusable for ANY loss -----------------------
def numeric_grad(loss_fn, X, y, w, b, eps=1e-6):
    grad_w = np.zeros_like(w)
    for j in range(len(w)):
        step = np.zeros_like(w); step[j] = eps
        grad_w[j] = (loss_fn(X, y, w + step, b) - loss_fn(X, y, w - step, b)) / (2 * eps)
    grad_b = (loss_fn(X, y, w, b + eps) - loss_fn(X, y, w, b - eps)) / (2 * eps)
    return grad_w, grad_b


def mae_loss(X, y, w, b):
    return float(np.abs(y - (X @ w + b)).mean())


def mae_grad_analytic(X, y, w, b):
    errors = y - (X @ w + b)
    return -(X.T @ np.sign(errors)) / len(y), -np.sign(errors).sum() / len(y)


gw_a, gb_a = mae_grad_analytic(X_check, y_check, w_check, b_check)
gw_n, gb_n = numeric_grad(mae_loss, X_check, y_check, w_check, b_check)
print("MAE  analytic:", gw_a.round(6), "\nMAE  numeric :", gw_n.round(6))
print("max diff:", np.abs(gw_a - gw_n).max())      # ~1e-9

# ---- 2) the broken gradient ---------------------------------------------------
def mse_grad_broken(X, y, w, b):
    errors = y - (X @ w + b)
    return -(X.T @ errors) / len(y) ** 2, -errors.sum() / len(y) ** 2

gw_b, _ = mse_grad_broken(X_check, y_check, w_check, b_check)
gw_n2, _ = numeric_grad(mse_loss, X_check, y_check, w_check, b_check)
print("\nbroken vs numeric, ratio:", (gw_b / gw_n2).round(3))     # all 1/7 -> caught

# ---- 3) the divergence threshold ---------------------------------------------
def diverges(eta):
    losses = AdalineGD(eta=eta, n_iter=20).fit(X, y).losses_
    return not np.isfinite(losses[-1]) or losses[-1] > losses[0]

print("\nbracket check -> diverges(1e-4):", diverges(1e-4), " diverges(0.5):", diverges(0.5))
low, high = 1e-4, 0.5                     # low converges, high diverges
for _ in range(40):
    mid = (low + high) / 2
    low, high = (low, mid) if diverges(mid) else (mid, high)
print(f"largest eta that still improves: about {low:.4f}")     # ~0.0496

# ---- 4) the contraction rate ------------------------------------------------
for eta in [0.1, 1.0, 1.5, 2.0, 2.5]:
    path = descend_1d(eta)
    observed = abs(path[-1] / path[0]) ** (1 / 12) if path[-1] != 0 else 0.0
    print(f"eta={eta:<4} |1-eta| = {abs(1 - eta):.3f}   observed rate = {observed:.3f}")
```

**2 — yes, the check catches it, and no, training does not obviously break.** Dividing by $n^2$ scales
every component of the gradient by the same $1/n$, so the *direction* is untouched — it is exactly
equivalent to training with `eta / n`. The loss still falls, just $n$ times more slowly, and you would
never notice from the curve. That is the whole argument for gradient checking: the bugs that survive are
the ones that look like a slightly worse hyperparameter.

**4 — they match exactly**, because $w_t = (1-\eta)^t w_0$ is the closed-form solution of this update, and
$\eta = 2$ gives a rate of exactly 1: perpetual oscillation.

</details>

---
<a id="s5"></a>
# 5 · Standardisation — Why a Single Learning Rate Was Never Going to Work

[⬆ back to TOC](#toc)

Look again at consequence 3 from §3: **the correction scales with the feature.**

$$\frac{\partial L}{\partial w_j} = -\frac{1}{n}\sum_i \bigl(y^{(i)} - \sigma(z^{(i)})\bigr)\,x_j^{(i)}$$

The same error produces a gradient proportional to $x_j$. A feature measured in centimetres and a feature
measured in millimetres therefore want learning rates 10× apart — but we have **one** $\eta$ for all of
them. So $\eta$ ends up set by the largest-scale feature, and every other weight learns too slowly.

The fix is to make the features comparable. **Standardisation** gives every feature mean 0 and standard
deviation 1:

$$x_j' = \frac{x_j - \mu_j}{\sigma_j}$$

> ⚠️ **Compute $\mu$ and $\sigma$ on the training set only**, then apply those same numbers to validation
> and test data. Using test statistics leaks information from data you are pretending not to have seen.
> (We have no split in this lab — Adaline has no capacity to overfit two features — but build the habit now.)

In [ ]:
import numpy as np

def standardise(X, mu=None, sigma=None):
    """
    Centre and scale features to mean 0, std 1.

    Args:
        X: Feature matrix, shape (n, m).
        mu: Optional pre-computed per-feature means (use the TRAINING ones on test data).
        sigma: Optional pre-computed per-feature standard deviations.

    Returns:
        (X_scaled, mu, sigma)
    """
    if mu is None:
        mu = X.mean(axis=0)                      # axis=0 -> one number PER FEATURE
    if sigma is None:
        sigma = X.std(axis=0)
        sigma = np.where(sigma == 0, 1.0, sigma)  # a constant feature would divide by zero
    return (X - mu) / sigma, mu, sigma


X_std, mu, sigma = standardise(X)

print("before:  mean =", X.mean(axis=0).round(3), " std =", X.std(axis=0).round(3))
print("after :  mean =", X_std.mean(axis=0).round(3), " std =", X_std.std(axis=0).round(3))
print()
print("ranges before:", (X.max(axis=0) - X.min(axis=0)).round(2))
print("ranges after :", (X_std.max(axis=0) - X_std.min(axis=0)).round(2))
print()
print("Standardisation does NOT change the shape of the cloud - it only re-labels the axes.")

### 🧪 Demo — What standardisation does to the loss surface

This is the picture worth remembering. Plot $L(w_1, w_2)$ (with $b$ at its optimum) for the raw and the
standardised features, and run gradient descent on both from the same starting point.

* **Raw features** → a long, thin valley. The gradient points across the valley rather than along it, so
  the path zig-zags and $\eta$ has to stay small enough for the *steep* direction.
* **Standardised features** → an almost circular bowl. The gradient points nearly at the minimum, and one
  learning rate suits both weights.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def loss_surface(X_data, y_data, w1_grid, w2_grid, b_fixed):
    """Evaluate the MSE loss over a grid of (w1, w2) with the bias held fixed."""
    surface = np.zeros_like(w1_grid)
    for i in range(w1_grid.shape[0]):
        for j in range(w1_grid.shape[1]):
            surface[i, j] = mse_loss(X_data, y_data,
                                     np.array([w1_grid[i, j], w2_grid[i, j]]), b_fixed)
    return surface


def gd_path(X_data, y_data, eta, steps, w_start, b_start=0.0):
    """Run full-batch GD and return the (steps+1, 2) path of the weights."""
    w, b = np.array(w_start, dtype=float), float(b_start)
    path = [w.copy()]
    for _ in range(steps):
        grad_w, grad_b = mse_grad_analytic(X_data, y_data, w, b)
        w -= eta * grad_w
        b -= eta * grad_b
        path.append(w.copy())
    return np.array(path)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

for ax, X_data, eta, title, span in [
        (axes[0], X,     0.0002, "RAW features: a thin valley, eta must stay tiny", 1.2),
        (axes[1], X_std, 0.30,   "STANDARDISED: a round bowl, eta can be large", 1.2)]:
    w1_grid, w2_grid = np.meshgrid(np.linspace(-span, span, 60), np.linspace(-span, span, 60))
    surface = loss_surface(X_data, y, w1_grid, w2_grid, b_fixed=0.5)
    contours = ax.contour(w1_grid, w2_grid, surface, levels=25, cmap="viridis", linewidths=0.8)
    path = gd_path(X_data, y, eta=eta, steps=40, w_start=[-1.0, 1.0], b_start=0.5)
    ax.plot(path[:, 0], path[:, 1], "o-", color="#d1495b", markersize=3, linewidth=1)
    ax.plot(path[0, 0], path[0, 1], "k*", markersize=12)
    ax.set(xlabel="w1 (sepal length)", ylabel="w2 (petal length)", title=f"{title}\neta={eta}")
    ax.set_aspect("equal")

plt.tight_layout(); plt.show()
print("Black star = the same starting point in both panels. Red = 40 gradient-descent steps.")
print("The contours on the left are stretched because feature 1 has a much larger scale than feature 2.")

### 🧪 Demo — Adaline on standardised features

Same class, same data, same 15 epochs — only the input scale changed. The learning rate is now
$\eta = 0.5$: **ten times the rate that exploded** on the raw features, and 5000× the one that crawled.
It is perfectly stable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

adaline = AdalineGD(eta=0.5, n_iter=15).fit(X_std, y)
axes[0].plot(range(1, 16), adaline.losses_, marker="o", color="#66a182")
axes[0].set(xlabel="epoch", ylabel="mean squared error / 2",
            title="Standardised features, eta = 0.5")

plot_decision_regions(X_std, y.astype(int), adaline.predict, ax=axes[1],
                      xlabel="sepal length [standardised]", ylabel="petal length [standardised]",
                      title="Adaline's boundary", class_names=["setosa", "versicolor"])
plt.tight_layout(); plt.show()

print(f"final loss     : {adaline.losses_[-1]:.4f}")
print(f"train accuracy : {(adaline.predict(X_std) == y).mean():.1%}")
print(f"weights        : w = {adaline.w_.round(3)}, b = {adaline.b_:+.3f}")
print()
print("Two things to notice:")
print("  * eta=0.5 is TEN TIMES the eta=0.05 that exploded on raw features, and here it is stable.")
print("  * the loss keeps falling after the accuracy hits 100%: Adaline minimises the LOSS,")
print("    not the error count, so it keeps pushing points away from the boundary.")

### ✍️ Exercise 5 — Scaling, done properly

1. Implement **min-max scaling**, $x' = (x - x_{\min}) / (x_{\max} - x_{\min})$, and train Adaline
   (`eta=0.5`, 15 epochs) on it. Compare the loss curve with standardisation. Both help — why?
2. **Train/test leakage, made visible.** Split the 100 examples into 70 train / 30 test with a shuffled
   index. Standardise (a) correctly, with `mu`/`sigma` from the training set, and (b) incorrectly, from the
   full dataset. Print both sets of `mu`/`sigma` and the test accuracy each way.
3. Multiply the *first* feature of `X` by 100 (pretend it was recorded in a different unit) and train on
   the raw result with `eta=0.0001`. Then standardise and retrain. Which of the two curves changed?
4. Take the model trained on `X_std` and predict for a **new** flower measuring 5.1 cm sepal / 1.4 cm
   petal. Remember it must be scaled with the *stored* `mu` and `sigma`.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) min-max scaling vs standardisation
# 2) correct vs leaking scaling on a 70/30 split
# 3) blow up feature 1 by 100x, before and after standardising
# 4) predict for a new flower - scale it with the STORED mu and sigma

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import numpy as np

# ---- 1) min-max --------------------------------------------------------------
X_minmax = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
fig, ax = plt.subplots()
for data, label in [(X_std, "standardised"), (X_minmax, "min-max")]:
    ax.plot(AdalineGD(eta=0.5, n_iter=15).fit(data, y).losses_, marker="o", label=label)
ax.set(xlabel="epoch", ylabel="loss", title="Both scalings tame the same problem")
ax.legend(); plt.tight_layout(); plt.show()

# ---- 2) leakage --------------------------------------------------------------
order = np.random.default_rng(0).permutation(len(y))
train_idx, test_idx = order[:70], order[70:]

# (a) correct: statistics from TRAIN only
X_tr, mu_tr, sd_tr = standardise(X[train_idx])
X_te, _, _ = standardise(X[test_idx], mu=mu_tr, sigma=sd_tr)
correct = AdalineGD(eta=0.5, n_iter=30).fit(X_tr, y[train_idx])
print("correct   -> mu", mu_tr.round(3), "test acc", (correct.predict(X_te) == y[test_idx]).mean())

# (b) leaking: statistics from the FULL dataset
X_leak, mu_all, sd_all = standardise(X)
leaking = AdalineGD(eta=0.5, n_iter=30).fit(X_leak[train_idx], y[train_idx])
print("leaking   -> mu", mu_all.round(3), "test acc",
      (leaking.predict(X_leak[test_idx]) == y[test_idx]).mean())

# ---- 3) a unit change -------------------------------------------------------
X_odd = X.copy(); X_odd[:, 0] *= 100
print("\nraw + 100x feature 1, eta=1e-4, final loss:",
      AdalineGD(eta=0.0001, n_iter=15).fit(X_odd, y).losses_[-1])
X_odd_std, _, _ = standardise(X_odd)
print("standardised, eta=0.5,   final loss:",
      round(AdalineGD(eta=0.5, n_iter=15).fit(X_odd_std, y).losses_[-1], 4))

# ---- 4) a new flower --------------------------------------------------------
new_flower = np.array([[5.1, 1.4]])
new_scaled = (new_flower - mu) / sigma
print("\nnew flower scaled:", new_scaled.round(3),
      "-> predicted class", adaline.predict(new_scaled)[0], "(0 = setosa)")
```

**1 — why both work.** Neither scaling is magic; both simply make the feature ranges *comparable*, which
turns the thin valley into a round bowl. Standardisation is the usual default because it is unaffected by
outliers in the way min-max is (one extreme value squashes everything else into a narrow band).

**2 — the accuracies barely differ here, and that is the point.** With two clean features and a linear
model, leakage costs you almost nothing measurable — which is exactly why it goes unnoticed until it
matters. On the *unchanged* test accuracy you learn nothing; on a small test set with heavy preprocessing
it can flatter your results substantially. Get the habit right while the stakes are zero.

**3 — the standardised curve is identical** to the one without the 100× (up to floating point):
standardisation divides that factor straight back out. The raw curve changes completely. Feature scaling
makes your training **invariant to the units your data was recorded in**.

</details>

---
<a id="s6"></a>
# 6 · Full-Batch vs Stochastic Gradient Descent

[⬆ back to TOC](#toc)

Adaline's update is an average over the **whole** training set:

$$\Delta\mathbf{w} = \frac{\eta}{n}\sum_{i=1}^{n}\bigl(y^{(i)} - \sigma(z^{(i)})\bigr)\mathbf{x}^{(i)}$$

Get the bookkeeping straight, because this is where students usually lose the thread:

> **One *step* = one update of $\mathbf{w}$ and $b$.**
> In full-batch gradient descent, **all $n$ examples must be visited before the weights may move once.**

With $n = 100$ that is fine. With ImageNet's $n \approx 1{,}300{,}000$ it is absurd: one and a half
million forward passes to earn a single step.

**Stochastic gradient descent (SGD)** estimates the same average from **one example** — or from a small
**mini-batch** — and steps immediately:

$$\Delta\mathbf{w} = \eta\,\bigl(y^{(i)} - \sigma(z^{(i)})\bigr)\mathbf{x}^{(i)}$$

Look closely: that is *algebraically the perceptron rule again*, with the continuous activation in place of
the threshold. It is **noisier** per step (one example is a poor estimate of an average over $n$),
**$n$ times cheaper** per step, and it enables **online learning** — new data can update the model as it
arrives, without a retrain.

| | Full batch | Mini-batch | Stochastic (SGD) |
|:--|:--|:--|:--|
| Examples per update | $n$ | e.g. 16–256 | 1 |
| Updates per epoch | 1 | $n / \text{batch}$ | $n$ |
| Gradient quality | exact | good | noisy |
| Vectorises well | yes | **yes** | poorly |
| Escapes flat regions / shallow minima | no | somewhat | yes (the noise helps) |
| Supports online learning | no | no | yes |

Mini-batch is what everybody actually uses — it keeps most of SGD's cheapness while still filling the
matrix multiply that GPUs are built for. `batch_size` in Lab 03's PyTorch code is exactly this number.

### 🧪 Demo — `AdalineSGD`: shuffling, and why it matters

Two implementation details are not optional:

* **Shuffle every epoch.** Iris arrives sorted by species. Without shuffling, SGD sees 50 setosa in a row,
  then 50 versicolor, and the weights swing with that ordering rather than converging.
* **Reduce the learning rate over time** (§6.2), because a constant $\eta$ leaves the weights permanently
  jittering around the minimum instead of settling into it.

In [ ]:
import numpy as np

class AdalineSGD:
    """
    Adaline trained with stochastic gradient descent - one update per EXAMPLE.

    Args:
        eta: Learning rate (constant unless adaptive_eta is set).
        n_iter: Number of epochs.
        shuffle: Reshuffle the training set before each epoch.
        adaptive_eta: If (c1, c2), use eta = c1 / (update_count + c2) instead of a constant.
        random_state: Seed for the initial weights and the shuffling.

    Attributes:
        w_, b_: Learned parameters.
        losses_: Average loss per epoch.
        update_count_: Total number of weight updates performed.
    """

    def __init__(self, eta=0.01, n_iter=15, shuffle=True, adaptive_eta=None, random_state=1):
        self.eta = eta
        self.n_iter = n_iter
        self.shuffle = shuffle
        self.adaptive_eta = adaptive_eta
        self.random_state = random_state
        self.w_initialised_ = False

    def _initialise_weights(self, m):
        self.rng_ = np.random.default_rng(self.random_state)
        self.w_ = self.rng_.normal(loc=0.0, scale=0.01, size=m)
        self.b_ = 0.0
        self.update_count_ = 0
        self.w_initialised_ = True

    def net_input(self, X):
        return X @ self.w_ + self.b_

    def activation(self, z):
        """Linear, exactly as in AdalineGD."""
        return z

    def predict(self, X):
        return np.where(self.activation(self.net_input(X)) >= 0.5, 1.0, 0.0)

    def _current_eta(self):
        if self.adaptive_eta is None:
            return self.eta
        c1, c2 = self.adaptive_eta
        return c1 / (self.update_count_ + c2)

    def _update(self, xi, target):
        """One example -> one update. Returns that example's loss."""
        output = self.activation(float(xi @ self.w_ + self.b_))
        error = target - output
        eta = self._current_eta()
        self.w_ += eta * error * xi          # compare with the perceptron rule: same shape!
        self.b_ += eta * error
        self.update_count_ += 1
        return 0.5 * error ** 2

    def fit(self, X, y):
        self._initialise_weights(X.shape[1])
        self.losses_ = []
        for _ in range(self.n_iter):
            if self.shuffle:
                order = self.rng_.permutation(len(y))
                X, y = X[order], y[order]
            losses = [self._update(xi, target) for xi, target in zip(X, y)]
            self.losses_.append(float(np.mean(losses)))
        return self

    def partial_fit(self, X, y):
        """Update on new data WITHOUT restarting - this is online learning."""
        if not self.w_initialised_:
            self._initialise_weights(X.shape[1])
        for xi, target in zip(np.atleast_2d(X), np.atleast_1d(y)):
            self._update(xi, target)
        return self


sgd = AdalineSGD(eta=0.01, n_iter=15).fit(X_std, y)
print(f"SGD: {sgd.update_count_} updates in {sgd.n_iter} epochs "
      f"({sgd.update_count_ // sgd.n_iter} per epoch, one per example)")
print(f"     final loss {sgd.losses_[-1]:.5f}, accuracy {(sgd.predict(X_std) == y).mean():.1%}")

batch = AdalineGD(eta=0.5, n_iter=15).fit(X_std, y)
print(f"\nBatch: {batch.n_iter} updates in {batch.n_iter} epochs (one per epoch)")
print(f"     final loss {batch.losses_[-1]:.5f}, accuracy {(batch.predict(X_std) == y).mean():.1%}")

In [ ]:
import matplotlib.pyplot as plt

no_shuffle = AdalineSGD(eta=0.01, n_iter=15, shuffle=False).fit(X_std, y)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(range(1, 16), sgd.losses_, marker="o", label="shuffled (correct)", color="#66a182")
axes[0].plot(range(1, 16), no_shuffle.losses_, marker="s", label="not shuffled", color="#d1495b")
axes[0].set(xlabel="epoch", ylabel="average loss", title="Shuffling is not optional")
axes[0].legend(fontsize=8)

plot_decision_regions(X_std, y.astype(int), sgd.predict, ax=axes[1],
                      xlabel="sepal length [std]", ylabel="petal length [std]",
                      title="SGD finds essentially the same line",
                      class_names=["setosa", "versicolor"])
plt.tight_layout(); plt.show()

print("Iris is sorted by species, so without shuffling SGD sees 50 setosa, then 50 versicolor.")
print("The weights chase whichever class it saw most recently.")

## 6.1 The fair comparison: per epoch vs per update

Comparing SGD and batch GD "per epoch" flatters SGD, because one SGD epoch contains 100 updates and one
batch epoch contains 1. Plot the same runs against both axes to see the real trade-off, and time them.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

def train_minibatch(X, y, batch_size, eta, n_epochs=15, random_state=1, adaptive=None):
    """
    One trainer for all three regimes: batch_size = n is full batch, 1 is SGD, anything else mini-batch.

    Args:
        X: Features (n, m).
        y: Targets (n,).
        batch_size: Examples per update.
        eta: Learning rate.
        n_epochs: Passes over the data.
        random_state: Seed.
        adaptive: Optional (c1, c2) for eta = c1 / (update + c2).

    Returns:
        dict with w, b, epoch_losses, update_losses, n_updates and seconds.
    """
    rng = np.random.default_rng(random_state)
    n, m = X.shape
    w, b = rng.normal(scale=0.01, size=m), 0.0
    epoch_losses, update_losses = [], []
    updates = 0
    start = time.perf_counter()

    for _ in range(n_epochs):
        order = rng.permutation(n)
        Xs, ys = X[order], y[order]
        for begin in range(0, n, batch_size):
            X_batch = Xs[begin:begin + batch_size]
            y_batch = ys[begin:begin + batch_size]
            current_eta = eta if adaptive is None else adaptive[0] / (updates + adaptive[1])

            errors = y_batch - (X_batch @ w + b)               # the gradient of THIS batch only
            w += current_eta * (X_batch.T @ errors) / len(y_batch)
            b += current_eta * errors.sum() / len(y_batch)

            updates += 1
            update_losses.append(mse_loss(X, y, w, b))         # full-data loss, for a fair curve
        epoch_losses.append(mse_loss(X, y, w, b))

    return {"w": w, "b": b, "epoch_losses": epoch_losses, "update_losses": update_losses,
            "n_updates": updates, "seconds": time.perf_counter() - start}


n = len(y)
runs = {
    f"full batch (n={n})": train_minibatch(X_std, y, batch_size=n,  eta=0.5,  n_epochs=15),
    "mini-batch (16)":     train_minibatch(X_std, y, batch_size=16, eta=0.5,  n_epochs=15),
    "SGD (1)":             train_minibatch(X_std, y, batch_size=1,  eta=0.05, n_epochs=15),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for label, run in runs.items():
    axes[0].plot(range(1, 16), run["epoch_losses"], marker="o", markersize=4, label=label)
    axes[1].plot(run["update_losses"], label=label)
axes[0].set(xlabel="epoch (one pass over the data)", ylabel="loss on the full data", yscale="log",
            title="Per EPOCH: small batches look far better")
axes[1].set(xlabel="weight update", ylabel="loss on the full data", yscale="log", xscale="log",
            title="Per UPDATE: the batch gradient is the best one")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'regime':<22}{'updates':>9}{'final loss':>13}{'seconds':>10}")
for label, run in runs.items():
    print(f"{label:<22}{run['n_updates']:>9}{run['epoch_losses'][-1]:>13.6f}{run['seconds']:>10.3f}")
print()
print("Read both panels together, and notice the noise in the right-hand SGD curve:")
print("  * per epoch, SGD wins - it made 100 updates while full batch made 1")
print("  * per update, full batch wins - its gradient is the exact average, not an estimate")
print("  * per second, mini-batch wins - the same work, done in vectorised chunks")

## 6.2 Adaptive learning rates

A constant $\eta$ has a built-in contradiction: large enough to make progress early is too large to settle
later, so SGD ends up jittering around the minimum forever. The classic fix is to shrink $\eta$ as
training proceeds:

$$\eta = \frac{c_1}{\text{number of updates} + c_2}$$

Big steps while far away, small steps once close. This is the ancestor of every learning-rate schedule you
will meet later (step decay, cosine annealing, warmup), and it is *separate* from the adaptive
**optimisers** (Adam, RMSProp) that scale each weight's step individually.

In [ ]:
import matplotlib.pyplot as plt

constant = train_minibatch(X_std, y, batch_size=1, eta=0.05, n_epochs=30)
decaying = train_minibatch(X_std, y, batch_size=1, n_epochs=30, eta=0.0, adaptive=(10.0, 200.0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(constant["update_losses"], label="constant eta = 0.05", alpha=0.8)
axes[0].plot(decaying["update_losses"], label="adaptive eta = 10 / (t + 200)", alpha=0.8)
axes[0].set(xlabel="update", ylabel="loss on the full data", yscale="log",
            title="Adaptive eta settles instead of jittering")
axes[0].legend(fontsize=8)

steps = np.arange(len(decaying["update_losses"]))
axes[1].plot(steps, np.full_like(steps, 0.05, dtype=float), label="constant")
axes[1].plot(steps, 10.0 / (steps + 200.0), label="10 / (t + 200)")
axes[1].set(xlabel="update", ylabel="learning rate", title="The schedule itself")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for label, run in [("constant", constant), ("adaptive", decaying)]:
    tail = np.array(run["update_losses"][-200:])
    print(f"{label:<9} final loss {run['epoch_losses'][-1]:.6f} | "
          f"jitter over the last 200 updates: std {tail.std():.1e}, range {np.ptp(tail):.1e}")
print()
print("Both schedules START at exactly eta = 10/200 = 0.05, so this is a fair comparison.")
print("The difference is the END: the constant run is still taking 0.05-sized steps fitted to")
print("single examples, so it never settles. The decayed run ends near 0.003 and the jitter")
print("collapses by two orders of magnitude - for a slightly lower final loss as well.")

### 🧪 Demo — Online learning with `partial_fit`

Because SGD needs only one example to make progress, a model can keep learning from a live stream. No
other regime in this notebook can do that: full-batch GD needs the whole dataset in memory to take a
single step.

In [ ]:
stream = np.random.default_rng(3).permutation(len(y))                 # arrival order of the data
first_60, remaining_40 = stream[:60], stream[60:]

online = AdalineSGD(eta=0.01, n_iter=5).fit(X_std[first_60], y[first_60])
print(f"after the initial fit on 60 examples : accuracy {(online.predict(X_std) == y).mean():.1%}")

# ... now the remaining 40 arrive one at a time, as they would from a sensor or a user
for index in remaining_40:
    online.partial_fit(X_std[index], y[index])

print(f"after 40 streamed single examples     : accuracy {(online.predict(X_std) == y).mean():.1%}")
print(f"total updates performed               : {online.update_count_}")
print(f"loss on all 100 flowers               : {mse_loss(X_std, y, online.w_, online.b_):.5f}")
print("\nNo retraining, no stored dataset - just one example at a time.")

### ✍️ Exercise 6 — Batch size is a hyperparameter

1. Run `train_minibatch` with `batch_size` in `[1, 4, 16, 32, 100]` at a fixed `eta=0.3`, 20 epochs.
   Plot the final loss against the batch size. Is bigger always better?
2. For each of those batch sizes, print `n_updates` and `seconds`. Where is the sweet spot on this
   (tiny) dataset, and why would the answer be different for MNIST in [§9](#s9)?
3. **The noise is real.** Plot the **last 200** `update_losses` for `batch_size=1` (40 epochs — 4000
   updates) and for `batch_size=100` (400 epochs, so it also has 400 updates to compare fairly). Which
   curve is smooth? Then re-run the `batch_size=1` case with `adaptive=(10.0, 200.0)` and see what
   happens to the noise.
4. **One learning rate, two verdicts.** Run `eta=1.0` for 20 epochs with `batch_size=100` and with
   `batch_size=1`. One of them is the best result in this whole section; the other blows up. Which way
   round is it, and why does the same $\eta$ behave so differently?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) sweep batch_size in [1, 4, 16, 32, 100] and plot the final loss
# 2) print n_updates and seconds for each
# 3) compare the tail noise of batch_size=1 and 100, then add the adaptive schedule
# 4) eta=1.5 at batch_size=100 vs batch_size=1

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

# ---- 1 and 2 -----------------------------------------------------------------
sizes = [1, 4, 16, 32, 100]
results = {s: train_minibatch(X_std, y, batch_size=s, eta=0.3, n_epochs=20) for s in sizes}

fig, ax = plt.subplots()
ax.plot(sizes, [results[s]["epoch_losses"][-1] for s in sizes], "o-")
ax.set(xlabel="batch size", ylabel="final loss", xscale="log", title="Final loss after 20 epochs")
plt.tight_layout(); plt.show()

print(f"{'batch':>6}{'updates':>10}{'final loss':>13}{'seconds':>10}")
for s in sizes:
    r = results[s]
    print(f"{s:>6}{r['n_updates']:>10}{r['epoch_losses'][-1]:>13.6f}{r['seconds']:>10.3f}")

# ---- 3) tail noise -----------------------------------------------------------
fig, ax = plt.subplots()
for size, epochs, label in [(1, 40, "SGD, constant eta"), (100, 400, "full batch")]:
    run = train_minibatch(X_std, y, batch_size=size, eta=0.3, n_epochs=epochs)
    ax.plot(run["update_losses"][-200:], label=label, alpha=0.8)
    print(f"{label:<20} tail std {np.std(run['update_losses'][-200:]):.2e}")
tamed = train_minibatch(X_std, y, batch_size=1, eta=0.0, n_epochs=40, adaptive=(10.0, 200.0))
ax.plot(tamed["update_losses"][-200:], label="SGD, adaptive eta", alpha=0.8)
print(f"{'SGD, adaptive eta':<20} tail std {np.std(tamed['update_losses'][-200:]):.2e}")
ax.set(xlabel="last 200 updates", ylabel="loss", title="Where the noise lives"); ax.legend()
plt.tight_layout(); plt.show()

# ---- 4) the same eta, two batch sizes ---------------------------------------
for s in [100, 1]:
    run = train_minibatch(X_std, y, batch_size=s, eta=1.0, n_epochs=20)
    print(f"batch={s:>3}  eta=1.0  final loss {run['epoch_losses'][-1]:.4e}")
```

**1 and 2 — the answer is not "smaller is better", and that is the useful surprise.** At the *fixed*
`eta=0.3` asked for, `batch_size=100` reaches the **lowest** loss (~0.008) and `batch_size=1` the
**highest** (~0.025), despite taking 100× as many steps. More updates only help if each update is
trustworthy, and 0.3 is far too large a step to take on the evidence of a single flower. Batch size and
learning rate are **not independent hyperparameters** — part 4 makes the same point from the other side.
On wall-clock, `batch_size=1` is already the slowest here (2000 tiny matrix products instead of 20 large
ones), and on MNIST (§9) that gap widens enormously. Hence mini-batch, not pure SGD, is what ships.

**3 — the SGD tail never stops moving.** Each update is fitted to a single example, so the loss on the
*full* dataset bounces — its tail standard deviation is orders of magnitude above the batch run's.
Adding the `c1/(t + c2)` schedule shrinks
the late steps and the bounce collapses by roughly a factor of 100. The full-batch tail is flat for a
different reason: it converged, and its exact gradient has nothing left to disagree about.

**4 — `eta=1.0` is the best setting in the section at `batch_size=100`, and diverges at `batch_size=1`.**
The full-batch gradient is the *exact* average over all 100 examples: it points genuinely downhill, so a
large step is safe and lands close to the minimum. A single-example gradient is a high-variance estimate
of that average — each step commits fully to one flower, and at $\eta = 1$ those over-corrections compound
instead of cancelling. Hence the standard rule of thumb, which you can now derive rather than memorise:
**shrink the batch, shrink the learning rate; grow the batch, grow the learning rate.** (In modern
practice that is the "linear scaling rule" used when a job moves to more GPUs.)

</details>

---
<a id="s7"></a>
# 7 · Multilayer Networks: Why Depth Needs Non-Linearity

[⬆ back to TOC](#toc)

## 7.1 The structure

A **multilayer feedforward** network (a *multilayer perceptron*, MLP): every unit is joined to **every**
unit of the next layer. Each link carries one **weight**; each unit adds one **bias**. A hidden unit $k$
computes

$$z^{(h)}_k = \sum_j w^{(h)}_{k,j}x_j + b^{(h)}_k, \qquad a^{(h)}_k = \sigma\bigl(z^{(h)}_k\bigr)$$

That is the entire content of a layer, whether it has 3 units or 3000 — **it is §1's neuron, repeated**.
More than one hidden layer makes the network **deep**, and training deep stacks needed special
algorithms; that is where the term *deep learning* comes from.

In [ ]:
mlp_diagram = """
flowchart LR
    subgraph IN["input layer: 4 features"]
        i1(("x1")); i2(("x2")); i3(("x3")); i4(("x4"))
    end
    subgraph H["hidden layer: 3 units, sigma"]
        h1(("a1")); h2(("a2")); h3(("a3"))
    end
    subgraph OUT["output layer: 3 classes, one-hot"]
        o1(("y1")); o2(("y2")); o3(("y3"))
    end
    i1 --> h1 & h2 & h3
    i2 --> h1 & h2 & h3
    i3 --> h1 & h2 & h3
    i4 --> h1 & h2 & h3
    h1 --> o1 & o2 & o3
    h2 --> o1 & o2 & o3
    h3 --> o1 & o2 & o3
"""
render_mermaid(mlp_diagram)

In [ ]:
import numpy as np

def count_parameters(layer_sizes):
    """
    Count the weights and biases of a fully connected network.

    Args:
        layer_sizes: e.g. [784, 50, 10] for input -> hidden -> output.

    Returns:
        (n_weights, n_biases)
    """
    weights = sum(a * b for a, b in zip(layer_sizes[:-1], layer_sizes[1:]))
    biases = sum(layer_sizes[1:])                 # the input layer has no bias
    return weights, biases


for sizes in [[2, 1], [4, 3, 3], [784, 50, 10], [784, 50, 50, 10], [784, 1000, 10]]:
    w_count, b_count = count_parameters(sizes)
    print(f"{str(sizes):<22} {w_count:>9,} weights + {b_count:>5,} biases = {w_count + b_count:>9,} learned numbers")

print("\nRow 3 is the MNIST network of section 9: 784*50 + 50*10 = 39,700 weights, plus 60 biases.")
print("Only the middle number (50) is a free choice - 784 and 10 are dictated by the data.")

## 7.2 One-hot output

Section 2 warned that the classes are **nominal**: encoding them as 0/1/2 invents an ordering. With one
output unit per class, the network never has to learn that "class 2 is twice class 1" — each class gets its
own unit, and the target is a vector with a single 1.

$$\text{label } 7 \;\longrightarrow\; [0,0,0,0,0,0,0,\mathbf{1},0,0]$$

In [ ]:
import numpy as np

def one_hot(y, n_classes):
    """
    Encode integer labels as one row per example with a single 1.

    Args:
        y: Integer labels, shape (n,).
        n_classes: Number of output units.

    Returns:
        Array of shape (n, n_classes), dtype float.
    """
    encoded = np.zeros((len(y), n_classes))
    encoded[np.arange(len(y)), y.astype(int)] = 1.0
    return encoded


labels = np.array([0, 2, 1, 2])
print("integer labels:", labels)
print("one-hot:")
print(one_hot(labels, 3))
print()
print("MNIST label 7 ->", one_hot(np.array([7]), 10)[0].astype(int))
print()
print("Decoding is argmax along axis 1:", one_hot(labels, 3).argmax(axis=1))

## 7.3 The learning procedure

1. **Forward-propagate** the training patterns through the network
2. **Calculate the loss**
3. **Backpropagate** it, obtaining the derivative with respect to **every** weight and bias
4. **Update** them

Backpropagation walks the *same graph* in the opposite direction, and the update happens only once the
signal has reached the front. **Why bother?** Because a multilayer network's loss is a complicated,
**non-convex** function of tens of thousands of parameters, and we need $\partial L / \partial w$ for each
of them without doing the algebra by hand once per weight. (Contrast Adaline: one convex MSE bowl, and the
derivative fitted on one line.)

In [ ]:
backprop_diagram = """
flowchart LR
    X[("X")] -->|"forward"| ZH["z_h = X W_h + b_h"] --> AH["a_h = sigma(z_h)"]
    AH --> ZO["z_out = a_h W_out + b_out"] --> AO["a_out = sigma(z_out)"] --> L["loss L(y, a_out)"]
    L -.->|"backward: dL/dz_out"| ZO
    ZO -.->|"dL/da_h"| AH
    AH -.->|"dL/dz_h"| ZH
    ZH -.->|"dL/dW_h"| U["update every W and b"]
    ZO -.->|"dL/dW_out"| U
"""
render_mermaid(backprop_diagram)

## 7.4 The linear collapse: why a non-linear $\sigma$ is not optional

Suppose the activations are the identity. Then

$$\mathbf{a}^{(\text{out})}
= W^{(\text{out})}\bigl(W^{(h)}\mathbf{x} + \mathbf{b}^{(h)}\bigr) + \mathbf{b}^{(\text{out})}
= \underbrace{W^{(\text{out})}W^{(h)}}_{W'}\,\mathbf{x}
+ \underbrace{W^{(\text{out})}\mathbf{b}^{(h)} + \mathbf{b}^{(\text{out})}}_{\mathbf{b}'}$$

$W'$ is **one matrix** and $\mathbf{b}'$ **one vector**, so the two-layer stack *is* a single linear layer,
with a straight decision boundary. A hundred layers is still one matrix. Adaline was linear and could not
do XOR; a stack of linear layers is the same model, so it cannot either.

Insert a non-linear $\sigma$ and it cannot be moved through the multiplication — no single $W'$ reproduces
$W^{(\text{out})}\sigma(W^{(h)}\mathbf{x})$. **That is the only reason depth buys anything.**

In [ ]:
import numpy as np

# Two random linear layers, 5 -> 8 -> 3
W_h = RNG.normal(size=(5, 8));  b_h = RNG.normal(size=8)
W_out = RNG.normal(size=(8, 3)); b_out = RNG.normal(size=3)
X_demo = RNG.normal(size=(4, 5))

two_layers = (X_demo @ W_h + b_h) @ W_out + b_out          # the honest forward pass

W_collapsed = W_h @ W_out                                   # (5, 3)  <- one matrix
b_collapsed = b_h @ W_out + b_out                           # (3,)    <- one vector
one_layer = X_demo @ W_collapsed + b_collapsed

print("two linear layers vs one collapsed layer:")
print("  identical to floating-point precision:", np.allclose(two_layers, one_layer))
print("  max absolute difference:", float(np.abs(two_layers - one_layer).max()))
print(f"  the 5x8 + 8x3 = {W_h.size + W_out.size} weights are worth exactly "
      f"{W_collapsed.size} independent numbers")

# The same stack WITH a non-linearity in the middle is a genuinely different function.
def sigmoid(z):
    """Logistic activation, clipped for numerical stability."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

with_sigma = sigmoid(X_demo @ W_h + b_h) @ W_out + b_out
print("\nwith sigmoid in the middle, still equal to a single linear layer?",
      np.allclose(with_sigma, one_layer))
print("difference:", float(np.abs(with_sigma - one_layer).max()), "<- a different function entirely")

### 🧪 Demo — XOR, solved by hand with one hidden layer

Exercise 1 proved that no single neuron computes XOR. Two layers can, and the weights are simple enough to
*design* rather than learn:

* hidden unit 1 = **OR**  ($\mathbf{w} = (1,1),\ b = -0.5$)
* hidden unit 2 = **AND** ($\mathbf{w} = (1,1),\ b = -1.5$)
* output = **h1 AND NOT h2** ($\mathbf{w} = (1,-1),\ b = -0.5$)

In words: *"at least one input is on, **and** not both."* The hidden layer **re-represents** the input in
coordinates where the problem has become linearly separable — that is what hidden layers are for. In §8
we stop designing these weights and start learning them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

W_hidden = np.array([[1.0, 1.0],       # rows = inputs, columns = hidden units
                     [1.0, 1.0]])
b_hidden = np.array([-0.5, -1.5])      # unit 1 = OR, unit 2 = AND
W_output = np.array([1.0, -1.0])       # h1 AND NOT h2
b_output = -0.5


def xor_network(X):
    """A hand-designed 2-2-1 network with unit-step activations."""
    hidden = unit_step(X @ W_hidden + b_hidden)      # (n, 2)
    return unit_step(hidden @ W_output + b_output)   # (n,)


print(" x1  x2 | h1(OR)  h2(AND) | out | XOR")
print("-" * 40)
for x1, x2 in INPUTS:
    hidden = unit_step(np.array([[x1, x2]]) @ W_hidden + b_hidden)[0]
    out = xor_network(np.array([[x1, x2]]))[0]
    print(f"  {x1}   {x2} |   {hidden[0]}       {hidden[1]}    |  {out}  |  {x1 ^ x2}")

xor_target = np.array([0, 1, 1, 0])
print("\nmatches XOR everywhere:", np.array_equal(xor_network(INPUTS), xor_target))

# The hidden layer moved the points; in ITS coordinates the problem is linearly separable.
hidden_coords = unit_step(INPUTS @ W_hidden + b_hidden).astype(float)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, coords, title, xlabel, ylabel in [
        (axes[0], INPUTS.astype(float), "Input space: NOT separable", "x1", "x2"),
        (axes[1], hidden_coords, "Hidden space: separable", "h1 (OR)", "h2 (AND)")]:
    for value, color, marker in [(0, "#3b6fb6", "o"), (1, "#d1495b", "s")]:
        pick = xor_target == value
        ax.scatter(coords[pick, 0], coords[pick, 1], c=color, marker=marker, s=140,
                   edgecolor="k", label=f"XOR = {value}", zorder=3)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=title, xlim=(-0.4, 1.4), ylim=(-0.4, 1.4))
    ax.legend(fontsize=8)
axes[1].plot([-0.4, 1.4], [0.9, -0.5], "k--", linewidth=1.2)      # the output unit's line
plt.tight_layout(); plt.show()

print("Note what happened on the right: (0,0) and (1,1) LANDED ON TOP OF EACH OTHER at (0,0) and (1,1),")
print("and the two 'XOR = 1' patterns collapsed into the single point (1,0). One line now suffices.")

### ✍️ Exercise 7 — Wiring, by hand and by counting

1. Design a hand-built 2-2-1 step network for **XNOR** (the negation of XOR: fire for $(0,0)$ and
   $(1,1)$). *Hint:* you only need to change the output unit.
2. Use `count_parameters` to answer: how many learned numbers does a `[784, 128, 64, 10]` network have?
   Which single layer holds most of them, and why?
3. **Verify the collapse yourself** for *three* stacked linear layers ($5 \to 8 \to 6 \to 3$): build the
   equivalent single $W'$ and $\mathbf{b}'$ and check with `np.allclose`.
4. Replace `unit_step` with `sigmoid` in `xor_network` (keep the same weights). Does it still classify all
   four patterns correctly after thresholding the output at 0.5? Multiply *all* weights and biases by 10
   and try again — what does that tell you about how sharp a sigmoid can be?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) XNOR with a hand-designed 2-2-1 network
# 2) count_parameters([784, 128, 64, 10])
# 3) collapse three linear layers into one
# 4) swap unit_step for sigmoid, then scale all parameters by 10

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import numpy as np

# ---- 1) XNOR = NOT XOR: flip the output unit --------------------------------
def xnor_network(X):
    hidden = unit_step(X @ W_hidden + b_hidden)              # same hidden layer: OR, AND
    return unit_step(hidden @ np.array([-1.0, 1.0]) + 0.5)   # NOT(h1) OR h2

print("XNOR:", xnor_network(INPUTS), " expected:", 1 - np.array([0, 1, 1, 0]))

# ---- 2) parameter count ------------------------------------------------------
w_count, b_count = count_parameters([784, 128, 64, 10])
print(f"\n[784,128,64,10]: {w_count:,} weights + {b_count} biases = {w_count + b_count:,}")
print("784*128 = 100,352 of them sit in the FIRST layer - the input is by far the widest thing here.")

# ---- 3) three linear layers collapse just the same --------------------------
W1, b1 = RNG.normal(size=(5, 8)), RNG.normal(size=8)
W2, b2 = RNG.normal(size=(8, 6)), RNG.normal(size=6)
W3, b3 = RNG.normal(size=(6, 3)), RNG.normal(size=3)
Xd = RNG.normal(size=(4, 5))

deep = ((Xd @ W1 + b1) @ W2 + b2) @ W3 + b3
W_all = W1 @ W2 @ W3
b_all = (b1 @ W2 + b2) @ W3 + b3
print("\nthree layers == one layer:", np.allclose(deep, Xd @ W_all + b_all))

# ---- 4) sigmoid instead of the step ----------------------------------------
def xor_sigmoid(X, scale=1.0):
    hidden = sigmoid(scale * (X @ W_hidden + b_hidden))
    return sigmoid(scale * (hidden @ W_output + b_output))

for scale in [1.0, 10.0]:
    out = xor_sigmoid(INPUTS, scale)
    print(f"\nscale={scale:>4}: raw output {out.round(3)}  ->  thresholded {(out >= 0.5).astype(int)}")
```

**4 — with `scale=1` it fails.** The sigmoid's hidden activations land near 0.3–0.8 rather than at a clean
0/1, and the output unit's line, designed for corners, no longer separates the smeared points. With
`scale=10` every sigmoid saturates and the network reproduces XOR exactly. So a **sigmoid with large
weights approximates a unit step** — the difference is that it is smooth, and therefore has a non-zero
derivative you can backpropagate through. That is the trade the whole of deep learning is built on, and
[§10](#s10) looks at the price (saturated sigmoids have almost no gradient left).

**Practical note for 1:** the hidden layer stayed the same for XOR *and* XNOR. Hidden layers learn
**re-representations** that many different output units can reuse — the same reason we can fine-tune a
pretrained network by replacing only its last layer.

</details>

---
<a id="s8"></a>
# 8 · A Neural Network from Scratch — Iris, All Three Classes

[⬆ back to TOC](#toc)

Everything now comes together. We build a **one-hidden-layer network** with nothing but NumPy:

```
4 features -> [ 8 hidden units, sigmoid ] -> [ 3 output units, sigmoid ] -> one-hot class scores
```

The pieces are all familiar:

| Piece | Where it came from |
|:--|:--|
| net input $z = Xw + b$ | §1 — the neuron |
| a differentiable activation | §4 — a flat derivative gives you nothing |
| MSE loss with the $\tfrac12$ | §4.2 |
| standardised inputs | §5 |
| mini-batches, shuffled each epoch | §6 |
| one-hot targets, one unit per class | §7.2 |
| a non-linear hidden layer | §7.4 — without it this is just Adaline |

**The one genuinely new thing is backpropagation**, and it is only the chain rule applied twice.

## 8.1 The two passes, written out

**Forward** (with $X$ of shape $(n, m)$, $W^{(h)}$ of shape $(m, h)$, $W^{(\text{out})}$ of shape $(h, k)$):

$$Z^{(h)} = XW^{(h)} + \mathbf{b}^{(h)}, \quad A^{(h)} = \sigma\bigl(Z^{(h)}\bigr), \quad
Z^{(\text{out})} = A^{(h)}W^{(\text{out})} + \mathbf{b}^{(\text{out})}, \quad
A^{(\text{out})} = \sigma\bigl(Z^{(\text{out})}\bigr)$$

**Loss** (MSE over classes, averaged over the batch, with the same $\tfrac12$ as §4):

$$L = \frac{1}{2n}\sum_{i=1}^{n}\sum_{c=1}^{k}\bigl(y_{ic} - a^{(\text{out})}_{ic}\bigr)^2$$

**Backward.** Work right to left, and carry one quantity per layer: $\delta = \partial L/\partial Z$.

$$\delta^{(\text{out})} = \underbrace{-\frac{1}{n}\bigl(Y - A^{(\text{out})}\bigr)}_{\partial L/\partial A^{(\text{out})}}
\;\odot\; \underbrace{A^{(\text{out})}\bigl(1 - A^{(\text{out})}\bigr)}_{\sigma'(Z^{(\text{out})})}
\qquad (n, k)$$

$$\frac{\partial L}{\partial W^{(\text{out})}} = A^{(h)\top}\delta^{(\text{out})}, \qquad
\frac{\partial L}{\partial \mathbf{b}^{(\text{out})}} = \sum_i \delta^{(\text{out})}_i$$

$$\delta^{(h)} = \bigl(\delta^{(\text{out})}W^{(\text{out})\top}\bigr) \odot \sigma'\bigl(Z^{(h)}\bigr)
\qquad (n, h)$$

$$\frac{\partial L}{\partial W^{(h)}} = X^{\top}\delta^{(h)}, \qquad
\frac{\partial L}{\partial \mathbf{b}^{(h)}} = \sum_i \delta^{(h)}_i$$

Three things to notice, and they are the whole idea:

* **$\delta^{(\text{out})}W^{(\text{out})\top}$ is the backward pass.** The forward pass multiplied by
  $W^{(\text{out})}$; the backward pass multiplies by its **transpose**. Same graph, opposite direction.
* Every gradient has the form `(inputs of that layer)ᵀ @ (delta of that layer)` — identical in shape to
  Adaline's `X.T @ errors`. It really is the same neuron.
* $\sigma'$ appears **once per layer**. If $\sigma'$ is small, the signal shrinks each time it passes
  one — that is the vanishing gradient problem, and [§10](#s10) measures it.

Useful shortcut used above: for the sigmoid, $\sigma'(z) = \sigma(z)\bigl(1-\sigma(z)\bigr) = a(1-a)$, so
the derivative is computed from the **activation we already stored**, with no extra exponentials.

In [ ]:
import numpy as np

def sigmoid(z):
    """Logistic activation, clipped so exp() cannot overflow."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


# Each entry is (activation, derivative-expressed-in-terms-of-the-ACTIVATION-OUTPUT).
# Storing the derivative that way is a standard trick: we already have 'a' from the forward pass.
ACTIVATIONS = {
    "sigmoid": (sigmoid,                       lambda a: a * (1.0 - a)),
    "tanh":    (np.tanh,                       lambda a: 1.0 - a ** 2),
    "relu":    (lambda z: np.maximum(0.0, z),  lambda a: (a > 0).astype(float)),
}


class NeuralNetMLP:
    """
    A one-hidden-layer fully connected network, forward and backward by hand.

    Args:
        n_features: Number of inputs, m.
        n_hidden: Number of hidden units, h - the only free choice in the architecture.
        n_classes: Number of output units, k (one per class).
        hidden_activation: Key into ACTIVATIONS for the hidden layer.
        random_state: Seed for the initial weights.

    Attributes:
        W_h, b_h: Hidden-layer parameters, shapes (m, h) and (h,).
        W_out, b_out: Output-layer parameters, shapes (h, k) and (k,).
    """

    def __init__(self, n_features, n_hidden, n_classes,
                 hidden_activation="sigmoid", random_state=1):
        rng = np.random.default_rng(random_state)
        self.n_classes = n_classes
        self.activation, self.activation_deriv = ACTIVATIONS[hidden_activation]
        self.hidden_activation = hidden_activation

        # Small random weights, scaled by 1/sqrt(fan_in): with many inputs, an unscaled
        # sum would saturate the sigmoid immediately and kill the gradient.
        self.W_h = rng.normal(0.0, 1.0 / np.sqrt(n_features), size=(n_features, n_hidden))
        self.b_h = np.zeros(n_hidden)
        self.W_out = rng.normal(0.0, 1.0 / np.sqrt(n_hidden), size=(n_hidden, n_classes))
        self.b_out = np.zeros(n_classes)

    # ---------------------------------------------------------------- forward
    def forward(self, X):
        """Return (z_h, a_h, z_out, a_out) - the backward pass needs the intermediates."""
        z_h = X @ self.W_h + self.b_h            # (n, h)
        a_h = self.activation(z_h)               # (n, h)
        z_out = a_h @ self.W_out + self.b_out    # (n, k)
        a_out = sigmoid(z_out)                   # (n, k)  output stays sigmoid: scores in (0, 1)
        return z_h, a_h, z_out, a_out

    def predict(self, X):
        """Predicted class indices, shape (n,) - one-hot decoded with argmax."""
        return self.forward(X)[3].argmax(axis=1)

    @staticmethod
    def loss(Y_onehot, a_out):
        """MSE over classes, averaged over examples, with the usual factor 1/2."""
        return float(((Y_onehot - a_out) ** 2).sum() / (2 * Y_onehot.shape[0]))

    # --------------------------------------------------------------- backward
    def backward(self, X, a_h, a_out, Y_onehot):
        """
        Backpropagate the loss and return the gradient of every parameter.

        Returns:
            (dW_h, db_h, dW_out, db_out), each shaped like its parameter.
        """
        n = X.shape[0]

        # ---- output layer -------------------------------------------------
        d_loss_d_a_out = -(Y_onehot - a_out) / n            # (n, k)  dL/dA_out
        delta_out = d_loss_d_a_out * a_out * (1.0 - a_out)  # (n, k)  x sigmoid'(z_out)

        dW_out = a_h.T @ delta_out                          # (h, k)  inputsᵀ @ delta
        db_out = delta_out.sum(axis=0)                      # (k,)

        # ---- hidden layer: the SAME graph, walked backwards ---------------
        delta_h = (delta_out @ self.W_out.T) * self.activation_deriv(a_h)   # (n, h)

        dW_h = X.T @ delta_h                                # (m, h)  inputsᵀ @ delta
        db_h = delta_h.sum(axis=0)                          # (h,)

        return dW_h, db_h, dW_out, db_out

    # ---------------------------------------------------------------- update
    def step(self, grads, eta):
        """Apply one gradient-descent update to every parameter."""
        dW_h, db_h, dW_out, db_out = grads
        self.W_h -= eta * dW_h
        self.b_h -= eta * db_h
        self.W_out -= eta * dW_out
        self.b_out -= eta * db_out


net = NeuralNetMLP(n_features=4, n_hidden=8, n_classes=3)
print("parameter shapes:")
for name in ["W_h", "b_h", "W_out", "b_out"]:
    print(f"  {name:<6} {str(getattr(net, name).shape):<10} "
          f"{getattr(net, name).size:>4} numbers")
print(f"  total  {sum(getattr(net, n).size for n in ['W_h', 'b_h', 'W_out', 'b_out'])} learned numbers")

### 🧪 Demo — Gradient-check the whole network

We wrote four gradient formulas by hand. **Check them**, exactly as in §4.3, but now over *every*
parameter of a deliberately tiny network. If this passes, the network is correct; if it does not, no amount
of hyperparameter tuning will save it.

In [ ]:
import numpy as np

def gradient_check(model, X, Y_onehot, eps=1e-6):
    """
    Compare every analytic gradient with a central finite difference.

    Args:
        model: A NeuralNetMLP.
        X: A small input batch.
        Y_onehot: Matching one-hot targets.
        eps: Finite-difference step.

    Returns:
        A dict mapping parameter name -> max relative difference.
    """
    _, a_h, _, a_out = model.forward(X)
    analytic = dict(zip(["W_h", "b_h", "W_out", "b_out"],
                        model.backward(X, a_h, a_out, Y_onehot)))
    report = {}

    for name in ["W_h", "b_h", "W_out", "b_out"]:
        param = getattr(model, name)
        numeric = np.zeros_like(param)
        iterator = np.nditer(param, flags=["multi_index"])
        while not iterator.finished:
            index = iterator.multi_index
            original = param[index]

            param[index] = original + eps
            loss_plus = model.loss(Y_onehot, model.forward(X)[3])
            param[index] = original - eps
            loss_minus = model.loss(Y_onehot, model.forward(X)[3])
            param[index] = original                     # always restore!

            numeric[index] = (loss_plus - loss_minus) / (2 * eps)
            iterator.iternext()

        scale = max(np.abs(analytic[name]).max(), np.abs(numeric).max(), 1e-12)
        report[name] = float(np.abs(analytic[name] - numeric).max() / scale)
    return report


tiny = NeuralNetMLP(n_features=3, n_hidden=4, n_classes=3, random_state=7)
X_tiny = RNG.normal(size=(5, 3))
Y_tiny = one_hot(RNG.integers(0, 3, size=5), 3)

print("max relative difference, analytic vs numeric:")
for name, difference in gradient_check(tiny, X_tiny, Y_tiny).items():
    verdict = "PASS" if difference < 1e-5 else "FAIL"
    print(f"  {name:<6} {difference:.3e}  {verdict}")

print("\nAll four formulas are verified against the definition of a derivative.")
print("31 parameters checked here; the same function will check 39,760 in section 9.")

## 8.2 The training loop

Mini-batch gradient descent, shuffled every epoch — §6, applied to a network instead of a single neuron.
We hold back 30 examples to check that the network generalises rather than memorises.

In [ ]:
import numpy as np

def train_mlp(model, X_train, y_train, X_valid=None, y_valid=None,
              eta=0.5, epochs=200, batch_size=16, random_state=1, log_every=None):
    """
    Train a NeuralNetMLP with mini-batch gradient descent.

    Args:
        model: The network to train, modified in place.
        X_train, y_train: Training features and INTEGER labels.
        X_valid, y_valid: Optional held-out set, evaluated once per epoch.
        eta: Learning rate.
        epochs: Passes over the training set.
        batch_size: Examples per weight update.
        random_state: Seed for the shuffling.
        log_every: Print a line every N epochs (None to stay quiet).

    Returns:
        A history dict of per-epoch lists.
    """
    rng = np.random.default_rng(random_state)
    Y_train = one_hot(y_train, model.n_classes)
    history = {"loss": [], "train_acc": [], "valid_acc": []}

    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(y_train))                 # shuffle EVERY epoch (section 6)
        for begin in range(0, len(order), batch_size):
            batch = order[begin:begin + batch_size]
            X_batch, Y_batch = X_train[batch], Y_train[batch]

            _, a_h, _, a_out = model.forward(X_batch)         # 1. forward-propagate
            grads = model.backward(X_batch, a_h, a_out, Y_batch)   # 2-3. loss, backpropagate
            model.step(grads, eta)                            # 4. update

        history["loss"].append(model.loss(Y_train, model.forward(X_train)[3]))
        history["train_acc"].append(float((model.predict(X_train) == y_train).mean()))
        if X_valid is not None:
            history["valid_acc"].append(float((model.predict(X_valid) == y_valid).mean()))

        if log_every and (epoch % log_every == 0 or epoch == 1):
            message = f"epoch {epoch:>4} | loss {history['loss'][-1]:.5f} | train {history['train_acc'][-1]:.1%}"
            if X_valid is not None:
                message += f" | valid {history['valid_acc'][-1]:.1%}"
            print(message)
    return history


# ---- the data: all 150 flowers, all 4 features, 3 classes ---------------------
y_int = np.array([list(SPECIES).index(name) for name in y_text])       # setosa/versicolor/virginica -> 0/1/2

order = np.random.default_rng(0).permutation(len(y_int))
train_idx, valid_idx = order[:120], order[120:]

X_train_raw, X_valid_raw = X_all[train_idx], X_all[valid_idx]
y_train, y_valid = y_int[train_idx], y_int[valid_idx]

X_train, mu_train, sigma_train = standardise(X_train_raw)             # statistics from TRAIN only
X_valid, _, _ = standardise(X_valid_raw, mu=mu_train, sigma=sigma_train)

print(f"train {X_train.shape}, valid {X_valid.shape}, classes {np.bincount(y_train)}")
print()

iris_net = NeuralNetMLP(n_features=4, n_hidden=8, n_classes=3, random_state=1)
iris_history = train_mlp(iris_net, X_train, y_train, X_valid, y_valid,
                         eta=0.5, epochs=300, batch_size=16, log_every=50)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(iris_history["loss"], color="#3b6fb6")
axes[0].set(xlabel="epoch", ylabel="MSE / 2", title="Loss - non-convex now, but it still descends")

axes[1].plot(iris_history["train_acc"], label="train", color="#3b6fb6")
axes[1].plot(iris_history["valid_acc"], label="validation (30 unseen flowers)", color="#d1495b")
axes[1].set(xlabel="epoch", ylabel="accuracy", title="Accuracy", ylim=(0.2, 1.02))
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"final train accuracy      : {iris_history['train_acc'][-1]:.1%}")
print(f"final validation accuracy : {iris_history['valid_acc'][-1]:.1%}")
print()
print("Per-class results on the validation set:")
predictions = iris_net.predict(X_valid)
for index, name in enumerate(SPECIES):
    pick = y_valid == index
    if pick.sum():
        print(f"  {name:<12} {int((predictions[pick] == index).sum())}/{int(pick.sum())} correct")

print("\nConfusion matrix (rows = truth, columns = prediction):")
confusion = np.zeros((3, 3), dtype=int)
for truth, prediction in zip(y_valid, predictions):
    confusion[truth, prediction] += 1
print("            " + "".join(f"{name[:9]:>11}" for name in SPECIES))
for index, name in enumerate(SPECIES):
    print(f"{name:<12}" + "".join(f"{value:>11}" for value in confusion[index]))

### 🧪 Demo — What the hidden layer bought us

Train the same network on just **two** features so the boundary can be drawn, and compare it with
Adaline's straight line. The hidden layer lets the boundary **bend**.

In [ ]:
import matplotlib.pyplot as plt

two_features = [2, 3]                                    # petal length, petal width
X_two, mu_two, sigma_two = standardise(X_all[:, two_features])

net_two = NeuralNetMLP(n_features=2, n_hidden=10, n_classes=3, random_state=1)
train_mlp(net_two, X_two, y_int, eta=0.5, epochs=400, batch_size=16)

net_tiny = NeuralNetMLP(n_features=2, n_hidden=1, n_classes=3, random_state=1)
train_mlp(net_tiny, X_two, y_int, eta=0.5, epochs=400, batch_size=16)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, model, title in [
        (axes[0], net_tiny, "1 hidden unit: barely more than a line"),
        (axes[1], net_two, "10 hidden units: curved, class-shaped regions")]:
    plot_decision_regions(X_two, y_int, model.predict, ax=ax, resolution=0.03,
                          xlabel="petal length [std]", ylabel="petal width [std]",
                          title=f"{title}\naccuracy {(model.predict(X_two) == y_int).mean():.1%}",
                          class_names=SPECIES)
plt.tight_layout(); plt.show()

print("The hidden layer's width controls how much the boundary can bend.")
print("Too little and it cannot fit; far too much and it will start fitting noise (Lab 04 territory).")

### ✍️ Exercise 8 — Tune, break, and diagnose the network

1. **Width.** Train with `n_hidden` in `[1, 2, 4, 8, 32]` (300 epochs, `eta=0.5`) and plot the final
   validation accuracy against the width. Where does adding units stop helping?
2. **Learning rate.** With `n_hidden=8`, try `eta` in `[0.01, 0.1, 0.5, 2.0, 10.0]`. Plot all five loss
   curves in one figure. Which diverges, and which is merely slow?
3. **Remove the non-linearity.** Add `"identity": (lambda z: z, lambda a: np.ones_like(a))` to
   `ACTIVATIONS` and compare it with `"sigmoid"` on **two** problems: Iris, and the XOR patterns
   (`INPUTS` with targets `[0, 1, 1, 0]`, `n_classes=2`, `eta=1.0`, 3000 epochs, `batch_size=4`). On which
   one does the collapse of §7.4 actually show up in the accuracy, and why?
4. **Does standardisation matter here?** Train on `X_train_raw` and compare the loss and the *saturation*
   of the hidden units — the fraction of `net.forward(X)[1]` above 0.99 or below 0.01. Then make the same
   comparison after multiplying feature 0 by 100 (`X_train_raw[:, 0] * 100`), which is what a unit change
   from cm to something smaller would do. Which comparison shows the effect clearly?
5. **Always be gradient-checking.** Change `delta_h` in `backward` to use `self.W_out` instead of
   `self.W_out.T` — sorry, that will not even run. Instead, drop the `* self.activation_deriv(a_h)` factor
   and re-run `gradient_check`. Does training still "work"? What accuracy does it reach?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) sweep n_hidden and plot the final validation accuracy
# 2) sweep eta and plot the loss curves
# 3) add an "identity" activation and compare
# 4) train on raw features, then measure hidden-unit saturation
# 5) sabotage backward(), then gradient_check() it

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import numpy as np
import matplotlib.pyplot as plt

# ---- 1) width ----------------------------------------------------------------
widths = [1, 2, 4, 8, 32]
accuracies = []
for width in widths:
    model = NeuralNetMLP(4, width, 3, random_state=1)
    history = train_mlp(model, X_train, y_train, X_valid, y_valid, eta=0.5, epochs=300)
    accuracies.append(history["valid_acc"][-1])
    print(f"n_hidden={width:>3}: valid {accuracies[-1]:.1%}")

fig, ax = plt.subplots()
ax.plot(widths, accuracies, "o-")
ax.set(xlabel="hidden units", ylabel="validation accuracy", xscale="log")
plt.tight_layout(); plt.show()

# ---- 2) learning rate --------------------------------------------------------
fig, ax = plt.subplots()
for eta in [0.01, 0.1, 0.5, 2.0, 10.0]:
    model = NeuralNetMLP(4, 8, 3, random_state=1)
    history = train_mlp(model, X_train, y_train, eta=eta, epochs=300)
    ax.plot(history["loss"], label=f"eta={eta} (final {history['loss'][-1]:.4f})")
ax.set(xlabel="epoch", ylabel="loss", yscale="log", title="Learning-rate sweep")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# ---- 3) no non-linearity -----------------------------------------------------
ACTIVATIONS["identity"] = (lambda z: z, lambda a: np.ones_like(a))

print("Iris (almost linearly separable):")
for activation in ["sigmoid", "identity"]:
    model = NeuralNetMLP(4, 8, 3, hidden_activation=activation, random_state=1)
    history = train_mlp(model, X_train, y_train, X_valid, y_valid, eta=0.5, epochs=300)
    print(f"  {activation:<9} valid {history['valid_acc'][-1]:.1%}")

print("XOR (needs a bend):")
X_xor, y_xor = INPUTS.astype(float), np.array([0, 1, 1, 0])
for activation in ["sigmoid", "identity"]:
    model = NeuralNetMLP(2, 4, 2, hidden_activation=activation, random_state=1)
    train_mlp(model, X_xor, y_xor, eta=1.0, epochs=3000, batch_size=4)
    print(f"  {activation:<9} accuracy {(model.predict(X_xor) == y_xor).mean():.0%} "
          f"predictions {model.predict(X_xor)}")

# ---- 4) raw features and saturation -----------------------------------------
def report(label, X_tr, X_va):
    model = NeuralNetMLP(4, 8, 3, random_state=1)
    history = train_mlp(model, X_tr, y_train, X_va, y_valid, eta=0.5, epochs=300)
    a_h = model.forward(X_tr)[1]
    print(f"  {label:<22} loss {history['loss'][-1]:.4f} | valid {history['valid_acc'][-1]:>6.1%} | "
          f"saturated {((a_h > 0.99) | (a_h < 0.01)).mean():.0%}")

print("\ncentimetres, as recorded:")
report("standardised", X_train, X_valid)
report("raw", X_train_raw, X_valid_raw)

wild_train, wild_valid = X_train_raw.copy(), X_valid_raw.copy()
wild_train[:, 0] *= 100
wild_valid[:, 0] *= 100
wild_std_train, mu_wild, sigma_wild = standardise(wild_train)
wild_std_valid, _, _ = standardise(wild_valid, mu=mu_wild, sigma=sigma_wild)
print("with feature 0 multiplied by 100:")
report("raw x100", wild_train, wild_valid)
report("standardised x100", wild_std_train, wild_std_valid)

# ---- 5) a sabotaged backward pass -------------------------------------------
class BrokenMLP(NeuralNetMLP):
    def backward(self, X, a_h, a_out, Y_onehot):
        n = X.shape[0]
        delta_out = -(Y_onehot - a_out) / n * a_out * (1 - a_out)
        delta_h = delta_out @ self.W_out.T           # <-- sigma'(z_h) factor DROPPED
        return X.T @ delta_h, delta_h.sum(0), a_h.T @ delta_out, delta_out.sum(0)

broken = BrokenMLP(3, 4, 3, random_state=7)
print("\ngradient check on the sabotaged network:")
for name, difference in gradient_check(broken, X_tiny, Y_tiny).items():
    print(f"  {name:<6} {difference:.3e}  {'PASS' if difference < 1e-5 else 'FAIL'}")

broken_full = BrokenMLP(4, 8, 3, random_state=1)
broken_history = train_mlp(broken_full, X_train, y_train, X_valid, y_valid, eta=0.5, epochs=300)
print(f"the broken network still reaches valid {broken_history['valid_acc'][-1]:.1%}")
```

**1 — the curve flattens almost immediately.** Iris needs very little capacity: 4 hidden units already do
as well as 32. One unit is a genuine bottleneck — three classes cannot be encoded in a single number that
the output layer can then separate reliably.

**2 — `eta=10.0` diverges or thrashes, `eta=0.01` has barely started after 300 epochs.** Exactly the §4.4
picture, now on a non-convex surface: there is no single "correct" $\eta$, only a usable range.

**3 — on Iris you cannot see the difference; on XOR you cannot miss it.** Both activations land around
96.7% on Iris, because Iris is *almost linearly separable* — a collapsed linear model is already nearly
optimal there, so removing the non-linearity costs nothing measurable. On XOR the identity network never gets past
50% (25–50% depending on the seed) no matter how long you train it — it is one linear layer, §7.4 and
Exercise 1 — while the sigmoid network reaches 100% for every seed. **The lesson about experiment design is the real payoff here:** a dataset that any
linear model solves cannot tell you anything about non-linearity, and picking the wrong benchmark is how
people convince themselves that an architectural change does nothing.

**4 — the same trap, again.** On centimetres, raw and standardised are indistinguishable (the loss is
within a few percent and validation accuracy is a coin-toss between 96.7% and 100% on only 30 flowers),
even though the raw run's hidden units are already ~29% saturated against ~20%. Multiply feature 0 by 100
and the picture is unambiguous: **100%** of the hidden activations saturate, and validation accuracy
collapses to roughly 27% — chance level. Once $z^{(h)}$ is far from 0, $\sigma'(z) = a(1-a)$ is
essentially 0, no gradient reaches the unit, and it never learns anything. Standardisation is insurance
against a failure mode that is invisible until it is total.

**5 — this is the most important part of the exercise.** The gradient check **fails loudly** on `W_h` and
`b_h` (`dW_out` is unaffected, which is a useful clue for localising the bug). But training still runs, the
loss still falls, and accuracy still lands somewhere plausible (around 73% here, against 96.7%)
— because the dropped factor
$a(1-a) \in (0, 0.25]$ is positive, so the corrupted "gradient" still usually points downhill. It is a
worse direction, not a nonsensical one. **You cannot detect this from the loss curve.** Gradient-check
every hand-written gradient.

</details>

---
<a id="s9"></a>
# 9 · The Same Network on MNIST — 784 → 50 → 10

[⬆ back to TOC](#toc)

This is the worked example of the week, end to end. **Nothing about the model changes** — we reuse
`NeuralNetMLP` and `train_mlp` untouched. Only the data is bigger:

```
label 7                -> one-hot, 10 entries
28x28 image  -> reshape -> 784 inputs
   -> net input + sigmoid -> 50 hidden
   -> net input + sigmoid -> 10 outputs -> class-membership scores
```

$784 \cdot 50 + 50 \cdot 10 = \mathbf{39{,}700}$ weights, plus $50 + 10 = 60$ biases. Every single one of
them is learned from the pixels. **Only the middle number (50) is a free choice** — 784 comes from the
image size and 10 from the number of digits.

### 🧪 Demo — Load the data

`load_mnist` tries several sources in turn so the notebook runs in Colab, in a local Jupyter, and offline:

1. `keras.datasets.mnist` — pre-installed in Colab and cached after the first call
2. a direct download of the same `mnist.npz`, needing nothing but NumPy
3. `sklearn.datasets.fetch_openml("mnist_784")`
4. `sklearn.datasets.load_digits` — bundled 8×8 digits, so **something** always works offline

Everything downstream reads `SIDE` from the loader, so the smaller fallback changes the pictures but not
the code.

In [ ]:
import os
import numpy as np

def load_mnist(n_train=10000, n_test=2000, random_state=0):
    """
    Load handwritten digits from whichever source is available.

    Args:
        n_train: Training examples to keep (subsampled to keep the lab fast).
        n_test: Held-out examples to keep.
        random_state: Seed for the subsampling.

    Returns:
        dict with X_train, y_train, X_test, y_test, side (image edge in pixels) and source.
        Pixels are floats in [0, 1]; labels are integers 0-9.
    """
    rng = np.random.default_rng(random_state)
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")       # quiet TensorFlow's banner

    def finish(X, y, side, source):
        X = X.reshape(len(X), -1).astype(np.float64) / 255.0 if X.max() > 1.5 else \
            X.reshape(len(X), -1).astype(np.float64)
        y = y.astype(int)
        pick = rng.permutation(len(y))
        train, test = pick[:n_train], pick[n_train:n_train + n_test]
        return {"X_train": X[train], "y_train": y[train],
                "X_test": X[test], "y_test": y[test], "side": side, "source": source}

    # 1) Keras (pre-installed in Colab, cached in ~/.keras after the first call)
    try:
        try:
            from keras.datasets import mnist
        except ImportError:
            from tensorflow.keras.datasets import mnist
        (X_a, y_a), (X_b, y_b) = mnist.load_data()
        return finish(np.concatenate([X_a, X_b]), np.concatenate([y_a, y_b]), 28, "keras.datasets.mnist")
    except Exception as error:
        print(f"[keras unavailable: {type(error).__name__}] trying a direct download ...")

    # 2) The same file, fetched with the standard library
    try:
        import urllib.request, tempfile
        path = os.path.join(tempfile.gettempdir(), "mnist.npz")
        if not os.path.exists(path):
            urllib.request.urlretrieve(
                "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", path)
        with np.load(path, allow_pickle=True) as data:
            X = np.concatenate([data["x_train"], data["x_test"]])
            y = np.concatenate([data["y_train"], data["y_test"]])
        return finish(X, y, 28, "direct download of mnist.npz")
    except Exception as error:
        print(f"[download failed: {type(error).__name__}] trying OpenML ...")

    # 3) OpenML
    try:
        from sklearn.datasets import fetch_openml
        bunch = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
        return finish(bunch.data, bunch.target.astype(int), 28, "openml mnist_784")
    except Exception as error:
        print(f"[OpenML failed: {type(error).__name__}] falling back to the bundled 8x8 digits ...")

    # 4) Always available, no network needed
    from sklearn.datasets import load_digits
    digits = load_digits()
    return finish(digits.data / 16.0, digits.target, 8, "sklearn load_digits (8x8 fallback)")


data = load_mnist(n_train=10000, n_test=2000)
SIDE = data["side"]
X_mnist_train, y_mnist_train = data["X_train"], data["y_train"]
X_mnist_test, y_mnist_test = data["X_test"], data["y_test"]

print(f"source          : {data['source']}")
print(f"X_train         : {X_mnist_train.shape}  ({SIDE}x{SIDE} = {SIDE * SIDE} inputs per image)")
print(f"X_test          : {X_mnist_test.shape}")
print(f"pixel range     : {X_mnist_train.min():.2f} to {X_mnist_train.max():.2f}  (already scaled to [0, 1])")
print(f"class counts    : {np.bincount(y_mnist_train)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 8, figsize=(11, 3.2))
for ax, index in zip(axes.ravel(), range(16)):
    ax.imshow(X_mnist_train[index].reshape(SIDE, SIDE), cmap="gray_r")
    ax.set_title(str(y_mnist_train[index]), fontsize=10)
    ax.axis("off")
fig.suptitle(f"Each image is a {SIDE}x{SIDE} grid of intensities, flattened into {SIDE * SIDE} inputs", y=1.04)
plt.tight_layout(); plt.show()

# One example, followed all the way through the encoding.
example = 0
label = y_mnist_train[example]
print(f"image {example}: label {label}")
print(f"  raw shape      : {SIDE} x {SIDE}")
print(f"  flattened      : {X_mnist_train[example].shape} -> the network's input layer")
print(f"  one-hot target : {one_hot(np.array([label]), 10)[0].astype(int)}")
print()
print("The flattening throws away every notion of 'next to' - the network is given "
      f"{SIDE * SIDE} unrelated\nnumbers and has to rediscover the spatial structure from data. "
      "Convolutions fix that, later in the course.")

### 🧪 Demo — Sanity-check the gradients at full size, then train

39,760 parameters, and a finite-difference check would need two forward passes for **each** of them. So we
check the *architecture* on a tiny batch first (cheap), and only then commit to training.

In [ ]:
import numpy as np

n_pixels = SIDE * SIDE
mnist_net = NeuralNetMLP(n_features=n_pixels, n_hidden=50, n_classes=10, random_state=1)

weights, biases = count_parameters([n_pixels, 50, 10])
print(f"architecture: {n_pixels} -> 50 -> 10")
print(f"  {weights:,} weights + {biases} biases = {weights + biases:,} learned numbers")

# Gradient check on a 4-example batch of the REAL data: the shapes are the real shapes,
# only the batch is small. This costs a few seconds instead of a few hours.
check_net = NeuralNetMLP(n_features=n_pixels, n_hidden=3, n_classes=10, random_state=3)
report = gradient_check(check_net, X_mnist_train[:4], one_hot(y_mnist_train[:4], 10))
print("\ngradient check (same code path, 3 hidden units so the check is affordable):")
for name, difference in report.items():
    print(f"  {name:<6} {difference:.3e}  {'PASS' if difference < 1e-5 else 'FAIL'}")

In [ ]:
import time

start = time.perf_counter()
mnist_history = train_mlp(mnist_net, X_mnist_train, y_mnist_train,
                          X_mnist_test, y_mnist_test,
                          eta=2.0, epochs=40, batch_size=100, log_every=5)
print(f"\ntrained {len(y_mnist_train):,} examples for 40 epochs in {time.perf_counter() - start:.1f} s "
      f"of pure NumPy")
print(f"weight updates performed: {40 * len(y_mnist_train) // 100:,}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(mnist_history["loss"], color="#3b6fb6")
axes[0].set(xlabel="epoch", ylabel="MSE / 2", title="Training loss")

axes[1].plot(mnist_history["train_acc"], label="train", color="#3b6fb6")
axes[1].plot(mnist_history["valid_acc"], label="held-out test", color="#d1495b")
axes[1].set(xlabel="epoch", ylabel="accuracy", title="Accuracy")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"final train accuracy : {mnist_history['train_acc'][-1]:.2%}")
print(f"final test  accuracy : {mnist_history['valid_acc'][-1]:.2%}")
print()
print("The gap between the two curves is the first hint of OVERFITTING - the network is starting to")
print("memorise. Regularisation, proper validation splits and early stopping are Lab 04's subject.")

### 🧪 Demo — Look at the mistakes, and at what the hidden units learned

Two diagnostics that cost nothing and are worth doing every time:

* **Inspect the errors.** Many are genuinely ambiguous digits — a useful reality check before chasing the
  last percent.
* **Visualise the first-layer weights.** Each hidden unit has one weight per pixel, so its weight vector
  can be reshaped back into an image: a picture of the pattern that unit responds to.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

predictions = mnist_net.predict(X_mnist_test)
wrong = np.flatnonzero(predictions != y_mnist_test)
print(f"{len(wrong)} of {len(y_mnist_test)} test images misclassified ({len(wrong) / len(y_mnist_test):.2%})")

fig, axes = plt.subplots(2, 8, figsize=(11, 3.2))
for ax, index in zip(axes.ravel(), wrong[:16]):
    ax.imshow(X_mnist_test[index].reshape(SIDE, SIDE), cmap="gray_r")
    ax.set_title(f"{y_mnist_test[index]}->{predictions[index]}", fontsize=9, color="#d1495b")
    ax.axis("off")
fig.suptitle("Misclassified: true -> predicted", y=1.04)
plt.tight_layout(); plt.show()

# Which digits get confused with which?
confusion = np.zeros((10, 10), dtype=int)
for truth, prediction in zip(y_mnist_test, predictions):
    confusion[truth, prediction] += 1

fig, ax = plt.subplots(figsize=(5.6, 4.8))
image = ax.imshow(np.log1p(confusion), cmap="Blues")
ax.set(xlabel="predicted", ylabel="true", xticks=range(10), yticks=range(10),
       title="Confusion matrix (log colour scale)")
for i in range(10):
    for j in range(10):
        if confusion[i, j]:
            ax.text(j, i, confusion[i, j], ha="center", va="center", fontsize=7,
                    color="white" if i == j else "black")
ax.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(11, 5.6))
limit = np.abs(mnist_net.W_h).max()
for index, ax in enumerate(axes.ravel()):
    if index < mnist_net.W_h.shape[1]:
        ax.imshow(mnist_net.W_h[:, index].reshape(SIDE, SIDE), cmap="RdBu_r",
                  vmin=-limit, vmax=limit)
        ax.set_title(f"unit {index}", fontsize=7)
    ax.axis("off")
fig.suptitle("Each hidden unit's weight vector, reshaped into an image.\n"
             "Red = 'ink here raises my activation', blue = 'ink here lowers it'.", y=1.02)
plt.tight_layout(); plt.show()

print("These are stroke and blob detectors: nobody designed them, gradient descent found them.")
print("They are the 'features' a classical pipeline would have hand-engineered - and that is the")
print("single most important idea in deep learning.")

### ✍️ Exercise 9 — Your turn on 39,700 weights

1. **Width.** Train `n_hidden` in `[10, 50, 200]` for 20 epochs each and compare test accuracy and
   wall-clock time. One of the three does something dramatic — when you see it, try that width again at
   `eta=0.5` before concluding anything about width.
2. **Batch size.** With `n_hidden=50`, compare `batch_size` in `[10, 100, 1000]` at 20 epochs. Time each.
   You should now see the effect §6 predicted for larger data — which is fastest *per epoch*, and which
   reaches the best accuracy *per second*?
3. **More data.** Reload with `n_train=30000` and retrain the 50-unit network. How much does test accuracy
   improve, and how much longer does an epoch take?
4. **Look for the overfitting.** Train for 150 epochs on only `n_train=1000` examples and plot train vs
   test accuracy. Where do the curves separate?
5. **Where the errors are.** From the confusion matrix, find the three most-confused digit pairs. Do they
   look plausible when you plot examples of them?

In [ ]:
# ✍️ YOUR CODE HERE
# 1) n_hidden in [10, 50, 200]: accuracy and seconds
# 2) batch_size in [10, 100, 1000]: accuracy and seconds
# 3) reload with n_train=30000 and retrain
# 4) n_train=1000, 150 epochs -> watch train and test separate
# 5) the three most-confused pairs, with pictures

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import time
import numpy as np
import matplotlib.pyplot as plt

n_pixels = SIDE * SIDE

# ---- 1) width, at two learning rates ----------------------------------------
for width in [10, 50, 200]:
    for eta in [2.0, 0.5]:
        model = NeuralNetMLP(n_pixels, width, 10, random_state=1)
        start = time.perf_counter()
        history = train_mlp(model, X_mnist_train, y_mnist_train, X_mnist_test, y_mnist_test,
                            eta=eta, epochs=20, batch_size=100)
        print(f"n_hidden={width:>4} eta={eta}: test {history['valid_acc'][-1]:.2%} "
              f"in {time.perf_counter() - start:.1f} s")

# ---- 2) batch size -----------------------------------------------------------
for batch in [10, 100, 1000]:
    model = NeuralNetMLP(n_pixels, 50, 10, random_state=1)
    start = time.perf_counter()
    history = train_mlp(model, X_mnist_train, y_mnist_train, X_mnist_test, y_mnist_test,
                        eta=2.0, epochs=20, batch_size=batch)
    print(f"batch={batch:>5}: test {history['valid_acc'][-1]:.2%} "
          f"in {time.perf_counter() - start:.1f} s "
          f"({20 * len(y_mnist_train) // batch:,} updates)")

# ---- 3) more data ------------------------------------------------------------
bigger = load_mnist(n_train=30000, n_test=2000)
model = NeuralNetMLP(n_pixels, 50, 10, random_state=1)
history = train_mlp(model, bigger["X_train"], bigger["y_train"],
                    bigger["X_test"], bigger["y_test"], eta=2.0, epochs=20, batch_size=100)
print(f"\n30k examples: test {history['valid_acc'][-1]:.2%}")

# ---- 4) overfitting on purpose ----------------------------------------------
small = load_mnist(n_train=1000, n_test=2000)
model = NeuralNetMLP(n_pixels, 50, 10, random_state=1)
history = train_mlp(model, small["X_train"], small["y_train"],
                    small["X_test"], small["y_test"], eta=2.0, epochs=150, batch_size=50)
fig, ax = plt.subplots()
ax.plot(history["train_acc"], label="train (1000 examples)")
ax.plot(history["valid_acc"], label="test")
ax.set(xlabel="epoch", ylabel="accuracy", title="Little data, plenty of parameters")
ax.legend(); plt.tight_layout(); plt.show()

# ---- 5) the most-confused pairs ---------------------------------------------
off_diagonal = confusion.copy()
np.fill_diagonal(off_diagonal, 0)
pairs = np.dstack(np.unravel_index(np.argsort(-off_diagonal.ravel()), off_diagonal.shape))[0][:3]
for truth, predicted in pairs:
    print(f"{truth} mistaken for {predicted}: {off_diagonal[truth, predicted]} times")
```

**1 — the surprise is that 200 units at `eta=2.0` collapses to about 44%**, far worse than 10 units. Width
is not a free parameter: a wider hidden layer feeds more terms into each output unit, so the gradients
arrive on a different scale and a learning rate tuned for 50 units is now too large. Re-run the 200-unit
network at `eta=0.5` and it recovers to about 90% — *still no better than 50 units at `eta=2.0`* (~93%),
for triple the time. So the honest conclusion has two parts: **width and learning rate must be tuned
together**, and once they are, this network's ceiling is set by its *architecture* — it cannot see that
pixels are neighbours — not by how many hidden units you give it.

**2 — `batch_size=10` reaches the highest accuracy per epoch but is by far the slowest in seconds**, and
`batch_size=1000` is fastest per epoch but learns least per epoch (only 10 updates). This is §6's trade-off
at a scale where it finally bites: 100 is a reasonable compromise, which is why numbers in the 32–256 range
are the default everywhere.

**3 — the single biggest win available here.** Tripling the data buys more accuracy than any
hyperparameter you can tune, and an epoch takes about three times as long. "Get more data" remains the
strongest move in practice.

**4 — the curves separate within a few dozen epochs.** 39,760 parameters against 1,000 examples is roughly
40 parameters per example, so the network can memorise. Train accuracy goes to 100% while test accuracy
stalls and then drifts down. That gap is the entire motivation for Lab 04.

**5 — expect pairs like 3↔5, 4↔9, 7↔9, 8↔3**; the exact ranking shifts with the seed and the subsample.
Plot the offending images and you will agree with the network more often than you expect.

</details>

---
<a id="s10"></a>
# 10 · Activation Functions

[⬆ back to TOC](#toc)

We have now met the activation in three roles, and every one of them was decided by its **derivative**:

* the **unit step** made the perceptron untrainable (derivative 0 wherever it exists)
* the **linear** activation gave Adaline a convex loss — and made depth pointless (§7.4)
* the **sigmoid** gave the hidden layer a non-linearity *with* a usable slope

So the choice of activation is not decoration. It is the first thing that matters in a deep network.

| Name | $\sigma(z)$ | Range | Derivative | In practice |
|:--|:--|:--|:--|:--|
| **Unit step** | $1$ if $z \ge 0$ else $0$ | $\{0, 1\}$ | $0$ (undefined at 0) | never trainable — historical only |
| **Linear** | $z$ | $\mathbb{R}$ | $1$ | Adaline; **regression output layers** |
| **Sigmoid** | $\dfrac{1}{1 + e^{-z}}$ | $(0, 1)$ | $\sigma(1-\sigma) \le 0.25$ | binary output layers; **saturates** in hidden layers |
| **Tanh** | $\dfrac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$ | $1 - \tanh^2 \le 1$ | a zero-centred sigmoid; better hidden unit, still saturates |
| **ReLU** | $\max(0, z)$ | $[0, \infty)$ | $1$ if $z>0$ else $0$ | **the modern default** for hidden layers |
| **Leaky ReLU** | $\max(\alpha z, z)$ | $\mathbb{R}$ | $1$ or $\alpha$ | ReLU without the permanently dead units |

Two properties to read off the table:

* **Zero-centred output.** Sigmoid outputs are all positive, so all the gradients feeding one weight vector
  share a sign and the weights are pushed together rather than independently. Tanh and ReLU do not have
  that problem to the same degree.
* **Maximum derivative.** The sigmoid's is $0.25$. Each layer multiplies the backpropagated signal by
  something $\le 0.25$ — the vanishing gradient, measured below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-5, 5, 1000)
LEAK = 0.1

FUNCTIONS = [
    ("unit step",  np.where(z >= 0, 1.0, 0.0),            np.zeros_like(z)),
    ("linear",     z,                                     np.ones_like(z)),
    ("sigmoid",    sigmoid(z),                            sigmoid(z) * (1 - sigmoid(z))),
    ("tanh",       np.tanh(z),                            1 - np.tanh(z) ** 2),
    ("ReLU",       np.maximum(0, z),                      (z > 0).astype(float)),
    ("leaky ReLU", np.where(z > 0, z, LEAK * z),          np.where(z > 0, 1.0, LEAK)),
]

fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for column, (name, values, derivative) in enumerate(FUNCTIONS):
    axes[0, column].plot(z, values, color="#3b6fb6")
    axes[0, column].set_title(name, fontsize=10)
    axes[1, column].plot(z, derivative, color="#d1495b")
    axes[1, column].set_title(f"d/dz  (max {derivative.max():.2f})", fontsize=9)
    for row in (0, 1):
        axes[row, column].axhline(0, color="grey", linewidth=0.6)
        axes[row, column].axvline(0, color="grey", linewidth=0.6)
        axes[row, column].set_xlim(-5, 5)
axes[0, 0].set_ylabel("sigma(z)")
axes[1, 0].set_ylabel("d sigma / dz")
fig.suptitle("Top: the activation. Bottom: its derivative - the row that decides whether training works.",
             y=1.03)
plt.tight_layout(); plt.show()

print("Read the bottom row:")
print("  unit step : flat zero -> a gradient carrying no information. The perceptron's dead end.")
print("  sigmoid   : peaks at 0.25 and vanishes beyond |z| > 4 -> saturated units stop learning.")
print("  tanh      : peaks at 1.00, and is zero-centred.")
print("  ReLU      : exactly 1 for z > 0 -> the signal passes through undiminished. And 0 below,")
print("              which is why a unit pushed permanently negative can DIE.")

### 🧪 Demo — Vanishing gradients, measured

Backpropagation multiplies by one $\sigma'$ per layer (§8.1). So the gradient that reaches layer 1 of an
$L$-layer network carries a factor of roughly

$$\prod_{\ell=1}^{L}\sigma'\bigl(z^{(\ell)}\bigr)$$

For sigmoids that product is **at most $0.25^L$** — and in practice smaller, since units are rarely at the
peak. This is a number you can simply compute, so let us compute it: draw typical pre-activations
$z \sim \mathcal{N}(0,1)$, take the average derivative per layer, and raise it to the power $L$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
depths = np.arange(1, 26)
z_typical = rng.normal(0.0, 1.0, size=200_000)          # typical, well-behaved pre-activations

DERIVATIVES = {
    "sigmoid": sigmoid(z_typical) * (1 - sigmoid(z_typical)),
    "tanh":    1 - np.tanh(z_typical) ** 2,
    "relu":    (z_typical > 0).astype(float),
}

fig, ax = plt.subplots()
for (name, derivative), color in zip(DERIVATIVES.items(), ["#3b6fb6", "#66a182", "#d1495b"]):
    mean_factor = derivative.mean()                     # attenuation contributed by ONE layer
    ax.plot(depths, mean_factor ** depths, "o-", markersize=3, color=color,
            label=f"{name} ({mean_factor:.2f} per layer)")
ax.plot(depths, 0.25 ** depths, "k--", linewidth=1, label="0.25^L (sigmoid's BEST case)")
ax.set(xlabel="number of layers, L", ylabel="surviving gradient factor", yscale="log",
       title="How much of the gradient reaches the first layer")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'activation':<10}{'max sigma1':>12}{'mean sigma1':>13}{'^5 layers':>12}{'^20 layers':>12}")
for name, derivative in DERIVATIVES.items():
    mean_factor = derivative.mean()
    print(f"{name:<10}{derivative.max():>12.3f}{mean_factor:>13.3f}"
          f"{mean_factor ** 5:>12.2e}{mean_factor ** 20:>12.2e}")

print()
print("Sigmoid: every layer multiplies the gradient by about 0.21, so after 20 layers roughly 1e-14")
print("of it survives - the first layer effectively never learns. Even at its theoretical best (0.25)")
print("it is hopeless. This is why deep sigmoid stacks were untrainable.")
print()
print("ReLU's 0.50 is a DIFFERENT KIND of number, and the distinction is the point: on a path where")
print("every unit is active the factor is exactly 1.00 - no attenuation at all, at any depth. The 0.5")
print("is the PROBABILITY that a unit is active, so ReLU either passes the gradient intact or blocks")
print("it. Attenuation cannot be undone; a blocked path merely makes the gradient sparse, and")
print("initialisation (the sqrt(2) of He init) compensates for the half that is switched off.")

### 🧪 Demo — Swapping the activation in *our* network

`NeuralNetMLP` already takes `hidden_activation`, so this is a one-word change. Same data, same seed, same
learning rate, three different hidden activations.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
summary = []

for activation, color in [("sigmoid", "#3b6fb6"), ("tanh", "#66a182"), ("relu", "#d1495b")]:
    model = NeuralNetMLP(n_features=4, n_hidden=16, n_classes=3,
                         hidden_activation=activation, random_state=1)
    history = train_mlp(model, X_train, y_train, X_valid, y_valid,
                        eta=0.5, epochs=300, batch_size=16)
    axes[0].plot(history["loss"], color=color, label=activation)
    axes[1].plot(history["train_acc"], color=color, label=activation)

    a_h = model.forward(X_train)[1]
    if activation == "relu":
        dead = float((a_h <= 0).all(axis=0).mean())        # units that never fire on ANY example
        note = f"{dead:.0%} of units are dead"
    else:
        # "Saturated" means the DERIVATIVE has died, so the test depends on the range:
        # sigmoid saturates near 0 and 1, tanh near -1 and +1.
        flat = (a_h > 0.99) | (a_h < 0.01) if activation == "sigmoid" else np.abs(a_h) > 0.99
        note = f"{flat.mean():.0%} of activations saturated"
    summary.append((activation, history["loss"][-1], history["valid_acc"][-1], note))

axes[0].set(xlabel="epoch", ylabel="loss", yscale="log", title="Loss")
axes[1].set(xlabel="epoch", ylabel="train accuracy", title="Accuracy", ylim=(0.2, 1.02))
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'activation':<12}{'final loss':>12}{'valid acc':>12}   note")
for activation, loss, accuracy, note in summary:
    print(f"{activation:<12}{loss:>12.5f}{accuracy:>11.1%}   {note}")
print()
print("On a 4-feature, one-hidden-layer problem all three work - the differences are small because")
print("there is only ONE layer for the gradient to squeeze through. The deeper the network, the more")
print("this choice decides whether it trains at all.")

### ✍️ Exercise 10 — Activations, hands on

1. **Add leaky ReLU** to `ACTIVATIONS` (use $\alpha = 0.01$) and train the Iris network with it.
   *Careful:* the derivative in `ACTIVATIONS` is expressed in terms of the **activation output** $a$, so
   write it as `np.where(a > 0, 1.0, alpha)`.
2. **Gradient-check your new activation** with `gradient_check`. Does it pass? (It should — and if it does
   not, your derivative is wrong, not your luck.)
3. **Kill some ReLUs.** Train the MNIST network with `hidden_activation="relu"` and `eta=2.0`, then count
   how many hidden units are dead (never positive on any training example). Retry with `eta=0.1`. What does
   the learning rate have to do with dead units?
4. **Saturation, watched over time.** For a sigmoid network trained on raw (unstandardised) Iris features,
   record the fraction of saturated hidden activations every 20 epochs. Plot it next to the loss.
5. **Why not a step function even now?** Add `"step": (unit_step, lambda a: np.zeros_like(a))` to
   `ACTIVATIONS`, train, and report what happens to the loss. Explain in one sentence.

In [ ]:
# ✍️ YOUR CODE HERE
# 1) leaky ReLU in ACTIVATIONS, then train
# 2) gradient_check it
# 3) count dead ReLU units at eta=2.0 and eta=0.1
# 4) track saturation over training on raw features
# 5) add a step activation and watch nothing happen

<details>
<summary>✅ <b>Show solution</b></summary>

```python
import numpy as np

# ---- 1 and 2) leaky ReLU -----------------------------------------------------
ALPHA = 0.01
ACTIVATIONS["leaky_relu"] = (lambda z: np.where(z > 0, z, ALPHA * z),
                             lambda a: np.where(a > 0, 1.0, ALPHA))

leaky = NeuralNetMLP(4, 16, 3, hidden_activation="leaky_relu", random_state=1)
history = train_mlp(leaky, X_train, y_train, X_valid, y_valid, eta=0.5, epochs=300)
print(f"leaky ReLU: valid {history['valid_acc'][-1]:.1%}")

check = NeuralNetMLP(3, 4, 3, hidden_activation="leaky_relu", random_state=7)
print("gradient check:", {k: f"{v:.1e}" for k, v in gradient_check(check, X_tiny, Y_tiny).items()})

# ---- 3) dead ReLUs -----------------------------------------------------------
n_pixels = SIDE * SIDE
for eta in [2.0, 0.1]:
    model = NeuralNetMLP(n_pixels, 50, 10, hidden_activation="relu", random_state=1)
    history = train_mlp(model, X_mnist_train, y_mnist_train, X_mnist_test, y_mnist_test,
                        eta=eta, epochs=15, batch_size=100)
    a_h = model.forward(X_mnist_train)[1]
    dead = int((a_h <= 0).all(axis=0).sum())
    print(f"eta={eta:<5} test {history['valid_acc'][-1]:.2%}, dead units {dead}/50")

# ---- 4) saturation over time ------------------------------------------------
model = NeuralNetMLP(4, 16, 3, random_state=1)
saturation, losses = [], []
for _ in range(15):
    history = train_mlp(model, X_train_raw, y_train, eta=0.5, epochs=20)
    a_h = model.forward(X_train_raw)[1]
    saturation.append(float(((a_h > 0.99) | (a_h < 0.01)).mean()))
    losses.append(history["loss"][-1])

fig, ax = plt.subplots()
ax.plot(np.arange(1, 16) * 20, saturation, "o-", label="fraction saturated")
ax.plot(np.arange(1, 16) * 20, losses, "s-", label="loss")
ax.set(xlabel="epoch", title="Raw features: saturation rises as the weights grow")
ax.legend(); plt.tight_layout(); plt.show()

# ---- 5) the step function ---------------------------------------------------
ACTIVATIONS["step"] = (unit_step, lambda a: np.zeros_like(a))
stepped = NeuralNetMLP(4, 16, 3, hidden_activation="step", random_state=1)
history = train_mlp(stepped, X_train, y_train, X_valid, y_valid, eta=0.5, epochs=300)
print(f"\nstep activation: loss {history['loss'][0]:.5f} -> {history['loss'][-1]:.5f}, "
      f"valid {history['valid_acc'][-1]:.1%}")
print("W_h changed at all?", not np.allclose(stepped.W_h,
      NeuralNetMLP(4, 16, 3, hidden_activation='step', random_state=1).W_h))
```

**3 — a large `eta` kills units.** One oversized update can drive a unit's weights so far negative that
$z \le 0$ for *every* input; from then on its derivative is 0, it receives no gradient, and it never
recovers. Small learning rates produce far fewer dead units — and leaky ReLU removes the failure mode
entirely, since its derivative is $\alpha$ rather than 0 on the negative side.

**5 — the hidden layer stops learning completely.** `W_out` and `b_out` still move (their gradient does not
involve $\sigma'(z_h)$), so the loss falls a little and then stops: you are training a single Adaline-style
output layer on top of a **frozen random binary encoding** of the input. `W_h` never changes, because every
gradient flowing into it is multiplied by zero. This is §4.1's point, now visible inside a real network: a
zero derivative is not a small gradient, it is *no information at all*.

</details>

---
<a id="s11"></a>
# 11 · Self-Check Quiz

[⬆ back to TOC](#toc)

Answer **without scrolling back up**. Aim for 8/10 before Lab 03. Every answer comes with an explanation
and the section to revisit.

In [ ]:
QUESTIONS = [
    {"q": "In z = w^T x + b, where does the bias b come from, historically?",
     "options": ["An extra input that is always 1",
                 "The firing threshold, moved to the other side of the inequality",
                 "The mean of the training labels",
                 "A regularisation term"],
     "answer": 1,
     "why": "sigma(z) >= theta was rewritten as z - theta >= 0 with b = -theta. (SS1.3)"},

    {"q": "The perceptron makes a CORRECT prediction on an example. What happens to w and b?",
     "options": ["They move a little towards the example",
                 "Nothing - both updates are exactly zero",
                 "Only the bias moves",
                 "They move away from the example"],
     "answer": 1,
     "why": "delta = eta * (y - y_hat) * x, and y - y_hat = 0. The rule learns only from mistakes. (SS3)"},

    {"q": "Which single change turns a perceptron into Adaline?",
     "options": ["Adding a hidden layer",
                 "Shuffling the data every epoch",
                 "Measuring the error from the continuous activation instead of the thresholded label",
                 "Using a smaller learning rate"],
     "answer": 2,
     "why": "One edge moves. That gives it a differentiable, convex loss to descend. (SS4.1)"},

    {"q": "Why can the perceptron not be trained by gradient descent?",
     "options": ["Its loss has many local minima",
                 "The unit step's derivative is zero wherever it exists",
                 "It has no bias term",
                 "Its features are not standardised"],
     "answer": 1,
     "why": "A zero gradient carries no direction information at all. (SS4.1, SS10)"},

    {"q": "For Adaline's MSE, what is dL/db?",
     "options": ["-(1/n) * sum_i (y_i - z_i) * x_i",
                 "-(1/n) * sum_i (y_i - z_i)",
                 "zero",
                 "-(1/n) * sum_i x_i"],
     "answer": 1,
     "why": "Same as dL/dw_j but with inner derivative -1 instead of -x_j: the bias has no input. (SS4.3)"},

    {"q": "You standardise your features. Which statistics do you use on the TEST set?",
     "options": ["The test set's own mean and std",
                 "The training set's mean and std",
                 "The mean and std of the whole dataset",
                 "It makes no difference"],
     "answer": 1,
     "why": "Anything else leaks information from data you are pretending not to have seen. (SS5)"},

    {"q": "In FULL-BATCH gradient descent with n = 1000, how many examples are visited before the "
          "weights move once?",
     "options": ["1", "32", "1000", "It depends on the learning rate"],
     "answer": 2,
     "why": "One step = one update, and the batch gradient averages over all n. (SS6)"},

    {"q": "You stack two linear layers, 5 -> 8 -> 3, with no non-linearity. What have you built?",
     "options": ["A network that can solve XOR",
                 "A single linear layer 5 -> 3",
                 "A network with 64 independent parameters",
                 "A convex optimisation problem with no solution"],
     "answer": 1,
     "why": "W_out W_h is one matrix. A hundred linear layers is still one matrix. (SS7.4)"},

    {"q": "A 784 -> 50 -> 10 network has how many WEIGHTS (ignoring biases)?",
     "options": ["844", "39,700", "39,760", "392,000"],
     "answer": 1,
     "why": "784*50 + 50*10 = 39,200 + 500 = 39,700. The 60 biases bring it to 39,760. (SS7.1, SS9)"},

    {"q": "Backpropagation exists in order to:",
     "options": ["Make the loss function convex",
                 "Compute dL/dw for every parameter of a non-convex network efficiently",
                 "Prevent overfitting",
                 "Choose the learning rate"],
     "answer": 1,
     "why": "It is the chain rule, organised so one backward sweep yields every derivative. (SS7.3, SS8.1)"},
]


def grade(answers):
    """Score a list of chosen option indices and explain every question."""
    correct = 0
    print("=" * 78)
    for number, (item, given) in enumerate(zip(QUESTIONS, answers), start=1):
        is_right = given == item["answer"]
        correct += int(is_right)
        print(f"Q{number}. {'CORRECT' if is_right else 'WRONG'}  -  {item['q']}")
        print(f"      you said  : {item['options'][given]}")
        if not is_right:
            print(f"      correct   : {item['options'][item['answer']]}")
        print(f"      why       : {item['why']}\n")
    print("=" * 78)
    print(f"SCORE: {correct}/{len(QUESTIONS)}")
    print("Excellent - you are ready for Lab 03 and PyTorch." if correct >= 8 else
          "Good start - revisit the sections named above, then retake the quiz.")
    return correct


try:
    import ipywidgets as widgets
    from IPython.display import display

    pickers, rows = [], []
    for number, item in enumerate(QUESTIONS, start=1):
        picker = widgets.RadioButtons(
            options=[(text, index) for index, text in enumerate(item["options"])],
            value=None, layout=widgets.Layout(width="95%"))
        pickers.append(picker)
        rows.append(widgets.VBox([widgets.HTML(f"<b>Q{number}. {item['q']}</b>"), picker]))

    button = widgets.Button(description="Check my answers", button_style="success", icon="check")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            unanswered = [i + 1 for i, p in enumerate(pickers) if p.value is None]
            if unanswered:
                print("Please answer every question. Still missing:", unanswered)
                return
            grade([p.value for p in pickers])

    button.on_click(on_click)
    display(widgets.VBox(rows + [button, output]))
except ImportError:
    print("ipywidgets is not available - answer on paper, then put your ten choices in a list")
    print("(one option index per question) and run:")
    print("    grade([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])   # <- replace with YOUR answers\n")
    for number, item in enumerate(QUESTIONS, start=1):
        print(f"Q{number}. {item['q']}")
        for index, option in enumerate(item["options"]):
            print(f"    {index}) {option}")
        print()

---
<a id="s12"></a>
# 12 · Summary, Cheat Sheet & Next Steps

[⬆ back to TOC](#toc)

## What you built

```mermaid
mindmap
  root((One neuron, four times))
    Perceptron 1957
      unit step
      learns only from mistakes
      no loss function
      converges only if separable
    Adaline 1960
      linear activation
      MSE, convex and differentiable
      gradient descent
      needs feature scaling
    Optimisation
      full batch, exact but slow
      SGD, noisy and cheap
      mini-batch, the practical choice
      adaptive learning rate
    Multilayer network
      hidden layer re-represents the input
      non-linearity or it collapses
      one-hot output
      backpropagation
```

In [ ]:
summary_diagram = """
flowchart TD
    N["one neuron<br/>z = wᵀx + b"] --> P["Perceptron<br/>unit step, no loss"]
    N --> A["Adaline<br/>linear, MSE loss"]
    P -->|"derivative is 0"| DEAD["cannot descend"]
    A -->|"convex bowl"| GD["gradient descent"]
    GD --> SC["standardise features"]
    GD --> SGD["SGD / mini-batch"]
    A -->|"stack it"| LIN["stacked linear layers"]
    LIN -->|"collapse into one matrix"| DEAD2["depth buys nothing"]
    LIN -->|"insert non-linear sigma"| MLP["multilayer network"]
    MLP --> BP["backpropagation"]
    BP --> MNIST["784 -> 50 -> 10<br/>39,760 parameters"]
"""
render_mermaid(summary_diagram)

## 📋 One-page cheat sheet

| Task | Code |
|:--|:--|
| net input for a whole batch | `z = X @ w + b` |
| unit step | `np.where(z >= 0, 1.0, 0.0)` |
| sigmoid, safely | `1 / (1 + np.exp(-np.clip(z, -500, 500)))` |
| sigmoid derivative from the output | `a * (1 - a)` |
| perceptron update | `w += eta * (y - y_hat) * x` |
| MSE loss | `((y - z) ** 2).sum() / (2 * n)` |
| MSE gradient | `grad_w = -(X.T @ (y - z)) / n` |
| one gradient-descent step | `w -= eta * grad_w` |
| standardise | `(X - X.mean(0)) / X.std(0)` — **fit on train only** |
| shuffle each epoch | `order = rng.permutation(n); X = X[order]` |
| mini-batches | `for i in range(0, n, bs): batch = order[i:i + bs]` |
| adaptive learning rate | `eta = c1 / (updates + c2)` |
| one-hot encode | `E = np.zeros((n, k)); E[np.arange(n), y] = 1` |
| one-hot decode | `probabilities.argmax(axis=1)` |
| a layer's gradient | `dW = inputs.T @ delta` |
| backprop through a layer | `delta_prev = (delta @ W.T) * sigma_prime` |
| numerical gradient check | `(L(w + eps) - L(w - eps)) / (2 * eps)` |
| count parameters | `sum(a * b for a, b in zip(sizes, sizes[1:])) + sum(sizes[1:])` |

## 🧠 The eight sentences worth memorising

1. The **bias is the threshold, moved** — nothing more.
2. One neuron draws **one straight line**; the weights rotate it, the bias slides it.
3. The perceptron **learns only from mistakes**, and only converges if a perfect line exists.
4. Adaline moved **one edge** — the error comes from the continuous activation — and that bought it a
   convex, differentiable **loss**.
5. A **zero derivative is not a small gradient; it is no information**.
6. **One step = one update.** Full batch visits all $n$ examples to earn it; SGD earns it from one.
7. **Standardise your features**, or your single learning rate will be wrong for every weight but one.
8. Without a **non-linearity**, a hundred layers collapse into one matrix.

## 🚀 Next steps

**Before Lab 03**, make sure you can do each of these *without looking anything up*:

- [ ] Write the perceptron update rule from memory, and say what happens on a correct prediction.
- [ ] Derive $\partial L/\partial w_j$ for Adaline's MSE, and then $\partial L/\partial b$.
- [ ] Write a numerical gradient check in five lines.
- [ ] Explain why the loss surface of raw Iris features is a thin valley and what standardisation does to it.
- [ ] State how many updates one epoch contains for `batch_size` = 1, 16 and $n$.
- [ ] Prove in two lines that two linear layers are one linear layer.
- [ ] Write the backward pass of a one-hidden-layer network, given the forward pass.
- [ ] Count the parameters of `[784, 50, 10]` in your head.

**Then keep going with:**

| Topic | Why it comes next | Where |
|:--|:--|:--|
| **PyTorch tensors & autograd** | everything you differentiated by hand today, done for you — and you will know exactly what it is doing | Lab 03 |
| **Softmax + cross-entropy** | the right output layer and loss for multi-class problems (we deliberately used sigmoid + MSE) | Lecture 03 |
| **Optimisers: momentum, Adam** | smarter uses of the same gradient | Lecture 04 |
| **Train / validation / test, regularisation** | the overfitting gap you saw in §9 | Lab 04 |
| **Convolutions** | exploiting the spatial structure that §9's `reshape` threw away | later |

> 🎓 **The one thing to carry forward.** In Lab 03 a network becomes three lines of PyTorch. That is a huge
> gain in convenience and *zero* gain in understanding — everything real is what you wrote today:
> a net input, an activation with a usable derivative, a loss, a gradient, and a step.
> When a PyTorch model refuses to learn, the bug is almost always in one of those five,
> and you now know what each of them looks like from the inside.

---

*DL2026 · Lab 02 · Basics of Neural Networks — reference: Raschka, Liu & Mirjalili,
**Machine Learning with PyTorch and Scikit-Learn**, chapters 2 and 11.*